# CLEF CheckThat! 2026 Task 2 — Attention-MLP + Ablation Study
## Qwen/Qwen3-Embedding-8B · Lightweight Ablation · Attention Comparison · Kaggle 2×T4

---

## 1. Project Overview

### Task — CLEF CheckThat! 2026 Task 2
Each **claim** must be classified as **True**, **False**, or **Conflicting** given a set of LLM-generated **reasoning traces**, each accompanied by an individual trace-level verdict.

### Objective
1. Embed the **claim** and each **reasoning trace** with **Qwen/Qwen3-Embedding-8B** (last-token pool, L2-normalised)
2. Apply **attention pooling** over trace embeddings → single representative trace vector
3. Compute a **disagreement vector** (`masked_std`) → captures cross-trace inconsistency
4. Extract **7 scalar verdict features** from the per-trace `Verdict_list` vote distribution
5. Concatenate all four groups and classify via a **weighted-loss MLP**

### Ablation Study (5 experiments)
| Experiment | Claim | Attn Pool | Disagreement | Verdict Feats |
|:---|:---:|:---:|:---:|:---:|
| A — Full Model | ✓ | ✓ | ✓ | ✓ |
| B — No Claim | ✗ | ✓ | ✓ | ✓ |
| C — No Attention | ✓ | ✗ | ✓ | ✓ |
| D — No Disagreement | ✓ | ✓ | ✗ | ✓ |
| E — No Verdict Features | ✓ | ✓ | ✓ | ✗ |

### Attention Comparison (3 pooling modes on full model)
Baseline Attention · Mean Pool · Max Pool

### Dataset
| Split | File | ID Key |
|:---|:---|:---|
| Train | `train.json` | `new_id` via `semantic_matches.json` |
| Validation | `validation.json` | Sequential index |
| Test | `test.json` | `query_id` |


## 2. Environment Setup

In [1]:
# Install required packages
!pip install "transformers>=4.40.0" accelerate bitsandbytes tqdm -q
print("Installation complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 75.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score,
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from collections import Counter
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

import warnings, os, gc, time, random, csv, copy
warnings.filterwarnings("ignore")

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU {_i}: {_p.name}  {_p.total_memory/1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
CUDA     : True
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


In [3]:
# ============================================================
# PATHS  ← adjust to your Kaggle input dataset
# ============================================================
DATA_BASE        = "/kaggle/input/clef-2026-checkthat-task-2-data"
TRAIN_JSON       = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/train.json"
VAL_JSON         = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/validation.json"
TEST_JSON        = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/test.json"
SEMANTIC_MATCHES = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/semantic_matches.json"

OUTPUT_DIR     = "/kaggle/working"
TRAIN_EMB_PATH = f"{OUTPUT_DIR}/train_qwen3_embeddings.npz"
VAL_EMB_PATH   = f"{OUTPUT_DIR}/val_qwen3_embeddings.npz"
TEST_EMB_PATH  = f"{OUTPUT_DIR}/test_qwen3_embeddings.npz"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# EMBEDDING MODEL CONFIG
# Verified from: https://huggingface.co/Qwen/Qwen3-Embedding-8B
# ============================================================
EMBEDDING_MODEL_ID   = "Qwen/Qwen3-Embedding-8B"
EMBEDDING_BATCH_SIZE = 4       # conservative for 16 GB T4
MAX_SEQ_LENGTH       = 8192   # Qwen3 native context window
NORMALIZE_EMBEDDINGS = True    # L2-normalise after last-token pool

# ============================================================
# DATA FIELD NAMES  ← adjust if your JSON uses different keys
# ============================================================
CLAIM_FIELD        = "claim"
TRACE_TEXT_FIELD   = "reasoning_traces"
VERDICT_LIST_FIELD = "Verdict_list"

# ============================================================
# LABEL MAPPING  (preserved from original)
# ============================================================
label_map         = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {0: "False", 1: "True", 2: "Conflicting"}

# ============================================================
# MODEL / TRAINING CONFIG  (preserved from original)
# ============================================================
BATCH_SIZE           = 16
MAX_ITERS            = 100
LEARNING_RATE        = 1e-3
EARLY_STOP_PATIENCE  = 20
WEIGHT_DECAY         = 1e-5
GRAD_CLIP_NORM       = 1.0
TRACE_MAX            = 20
SCALAR_DIM           = 7
ATTN_HIDDEN          = 256
HIDDEN_DIMS          = [512, 256, 128]
OUTPUT_DIM           = 3
DEVICE               = "cuda" if torch.cuda.is_available() else "cpu"

EMB_DIM      = 4096
COMBINED_DIM = EMB_DIM * 3 + SCALAR_DIM

print("=" * 62)
print(f"Device        : {DEVICE}  ({torch.cuda.device_count()} GPU(s))")
print(f"Emb model     : {EMBEDDING_MODEL_ID}")
print(f"EMB_DIM       : {EMB_DIM}")
print(f"COMBINED_DIM  : {COMBINED_DIM}  (3×EMB_DIM + SCALAR_DIM)")
print(f"HIDDEN_DIMS   : {HIDDEN_DIMS}")
print("=" * 62)

Device        : cuda  (2 GPU(s))
Emb model     : Qwen/Qwen3-Embedding-8B
EMB_DIM       : 4096
COMBINED_DIM  : 12295  (3×EMB_DIM + SCALAR_DIM)
HIDDEN_DIMS   : [512, 256, 128]


## 3. Reproducibility Configuration

In [4]:
RANDOM_SEED   = 42
DETERMINISTIC = True

def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False

set_seed()

EXPERIMENT_META = {
    "task": "CLEF CheckThat 2026 Task 2",
    "embedding_model": EMBEDDING_MODEL_ID,
    "random_seed": RANDOM_SEED,
    "deterministic": DETERMINISTIC,
    "batch_size": BATCH_SIZE,
    "max_iters": MAX_ITERS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "hidden_dims": HIDDEN_DIMS,
    "trace_max": TRACE_MAX,
    "scalar_dim": SCALAR_DIM,
    "emb_dim": EMB_DIM,
    "combined_dim": COMBINED_DIM,
}
with open(f"{OUTPUT_DIR}/experiment_meta.json", "w") as _f:
    json.dump(EXPERIMENT_META, _f, indent=2)

print(f"Seed {RANDOM_SEED} set  |  deterministic={DETERMINISTIC}")
print(f"Metadata logged → {OUTPUT_DIR}/experiment_meta.json")

Seed 42 set  |  deterministic=True
Metadata logged → /kaggle/working/experiment_meta.json


## 4. Dataset Loading

In [5]:
def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def numpy_to_python(obj):
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, dict):          return {k: numpy_to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [numpy_to_python(i) for i in obj]
    return obj

print("Loading JSON datasets...")
_train_data       = load_json(TRAIN_JSON)
_val_data         = load_json(VAL_JSON)
_test_data        = load_json(TEST_JSON)
_semantic_matches = load_json(SEMANTIC_MATCHES)

print(f"Train items       : {len(_train_data)}")
print(f"Val items         : {len(_val_data)}")
print(f"Test items        : {len(_test_data)}")
print(f"Semantic matches  : {len(_semantic_matches)}")

if _train_data:       print(f"\nTrain keys : {list(_train_data[0].keys())}")
if _val_data:         print(f"Val keys   : {list(_val_data[0].keys())}")
if _test_data:        print(f"Test keys  : {list(_test_data[0].keys())}")
if _semantic_matches: print(f"Match keys : {list(_semantic_matches[0].keys())}")

Loading JSON datasets...
Train items       : 6400
Val items         : 1600
Test items        : 2558
Semantic matches  : 6469

Train keys : ['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'label', 'verdict']
Val keys   : ['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'label', 'verdict']
Test keys  : ['Reasoning_traces', 'Verdict_list', 'claim', 'evidences', 'Label', 'verdict', 'query_id']
Match keys : ['new_id', 'old_id', 'similarity', 'new_claim', 'old_claim']


In [6]:
# ============================================================
# DATASET MAPPING  (preserved exactly from original notebook)
# ============================================================

def create_train_id_to_item() -> dict:
    claim_to_item  = {item[CLAIM_FIELD]: item for item in _train_data}
    new_id_to_item = {}
    for match in _semantic_matches:
        new_claim = match["new_claim"]
        if new_claim in claim_to_item:
            new_id_to_item[match["new_id"]] = claim_to_item[new_claim]
    print(f"Train ID→item  : {len(new_id_to_item)} entries mapped")
    return new_id_to_item

def create_val_id_to_item() -> dict:
    mapping = {idx: item for idx, item in enumerate(_val_data)}
    print(f"Val ID→item    : {len(mapping)} entries")
    return mapping

def create_test_id_to_item() -> dict:
    mapping = {item["query_id"]: item for item in _test_data}
    print(f"Test ID→item   : {len(mapping)} entries")
    return mapping

train_id_to_item = create_train_id_to_item()
val_id_to_item   = create_val_id_to_item()
test_id_to_item  = create_test_id_to_item()

Train ID→item  : 6400 entries mapped
Val ID→item    : 1600 entries
Test ID→item   : 2558 entries


## 5. Qwen3-Embedding-8B Embedding Generation

### Pooling Strategy (verified from official model card)
Qwen3-Embedding-8B is a **decoder-only** transformer (Qwen3 backbone).  
The official recommended pooling for decoder-only embedding models is **last-token pooling**:  
the hidden state of the **last non-padding token** (the EOS position) is used as the sentence embedding.

### Normalisation
**L2 normalisation** is applied after pooling — following the official Qwen3-Embedding guidance.

### Tokeniser Settings (verified from model)
| Setting | Value | Source |
|:---|:---|:---|
| `padding` | `True` (right-padded) | tokenizer default |
| `truncation` | `True` | model card |
| `max_length` | `8192` | Qwen3 context window |
| `return_tensors` | `"pt"` | PyTorch |

### Embedding Storage Format
Each item is stored under key `"id_{item_id}"` in the `.npz` archive:

| Row | Content |
|:---|:---|
| 0 | Claim embedding `(EMB_DIM,)` |
| 1..N | Reasoning trace embeddings `(num_traces, EMB_DIM)` |

Stacked shape: `(1 + num_traces, EMB_DIM)` — same format expected by `split_and_pad`.


In [7]:
print(f"Loading tokeniser : {EMBEDDING_MODEL_ID}")
_emb_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_ID, trust_remote_code=True)

print(f"Loading model     : {EMBEDDING_MODEL_ID}  (torch.float16, device_map=auto)")
_emb_model = AutoModel.from_pretrained(
    EMBEDDING_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
_emb_model.eval()

_actual_emb_dim = _emb_model.config.hidden_size
print(f"\nmodel.config.hidden_size : {_actual_emb_dim}")
if _actual_emb_dim != EMB_DIM:
    print(f"  ⚠  EMB_DIM was {EMB_DIM}; updating to {_actual_emb_dim}")
    EMB_DIM      = _actual_emb_dim
    COMBINED_DIM = EMB_DIM * 3 + SCALAR_DIM
    print(f"  Updated COMBINED_DIM → {COMBINED_DIM}")
else:
    print(f"  ✓ EMB_DIM = {EMB_DIM} confirmed")

print(f"\nTokeniser padding_side    : {_emb_tokenizer.padding_side}")
print(f"Tokeniser model_max_length: {_emb_tokenizer.model_max_length}")
print(f"Using max_length          : {MAX_SEQ_LENGTH}")

Loading tokeniser : Qwen/Qwen3-Embedding-8B


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading model     : Qwen/Qwen3-Embedding-8B  (torch.float16, device_map=auto)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


model.config.hidden_size : 4096
  ✓ EMB_DIM = 4096 confirmed

Tokeniser padding_side    : right
Tokeniser model_max_length: 131072
Using max_length          : 8192


In [8]:
# ============================================================
# EMBEDDING HELPER FUNCTIONS
# ============================================================

def last_token_pool(last_hidden_states: torch.Tensor,
                    attention_mask:     torch.Tensor) -> torch.Tensor:
    """Last non-padding token pool — official Qwen3-Embedding strategy."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size       = last_hidden_states.shape[0]
    return last_hidden_states[
        torch.arange(batch_size, device=last_hidden_states.device),
        sequence_lengths,
    ]

def _get_trace_text(trace) -> str:
    if isinstance(trace, str):
        return trace
    if isinstance(trace, dict):
        for key in ("text", "content", "response", "reasoning", "trace", "passage"):
            if key in trace:
                return str(trace[key])
        return str(trace)
    return str(trace)

def get_item_texts(item: dict):
    claim_text  = item.get(CLAIM_FIELD, item.get("claim_text", "")).strip()
    traces_raw  = item.get(TRACE_TEXT_FIELD, [])
    trace_texts = [_get_trace_text(t) for t in traces_raw]
    return claim_text, trace_texts

@torch.no_grad()
def encode_texts(texts: list,
                 batch_size: int  = EMBEDDING_BATCH_SIZE,
                 max_length: int  = MAX_SEQ_LENGTH,
                 normalize:  bool = NORMALIZE_EMBEDDINGS) -> np.ndarray:
    """Encode texts → (N, EMB_DIM) float32 using Qwen3-Embedding-8B."""
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch   = texts[start: start + batch_size]
        encoded = _emb_tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        first_device = next(_emb_model.parameters()).device
        encoded = {k: v.to(first_device) for k, v in encoded.items()}
        outputs    = _emb_model(**encoded)
        embeddings = last_token_pool(outputs.last_hidden_state, encoded["attention_mask"])
        if normalize:
            embeddings = F.normalize(embeddings.float(), p=2, dim=-1)
        else:
            embeddings = embeddings.float()
        all_embeddings.append(embeddings.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

# Generate & Cache Embeddings

In [9]:
def generate_embeddings_for_split(id_to_item, id_order, split_name, cache_path):
    """
    Row 0   = claim embedding  (EMB_DIM,)
    Rows 1+ = trace embeddings (num_traces, EMB_DIM)
    Stored as plain dict (NOT NpzFile) — required for DataLoader thread safety.
    """
    if os.path.exists(cache_path):
        print(f"[{split_name}] Cache found — loading from {cache_path}")
        # ⚠ dict() is REQUIRED: NpzFile lazy readers are NOT thread-safe with DataLoader workers
        return dict(np.load(cache_path, allow_pickle=False))

    print(f"\n[{split_name}] Generating embeddings for {len(id_order)} items …")
    emb_dict = {}

    for id_ in tqdm(id_order, desc=f"[{split_name}]"):
        item = id_to_item[id_]
        claim_text, trace_texts = get_item_texts(item)

        if not claim_text:   claim_text   = "[EMPTY CLAIM]"
        if not trace_texts:  trace_texts  = ["[NO TRACE]"]

        claim_emb  = encode_texts([claim_text],  normalize=NORMALIZE_EMBEDDINGS)
        trace_embs = encode_texts(trace_texts,   normalize=NORMALIZE_EMBEDDINGS)

        stacked = np.vstack([claim_emb, trace_embs]).astype(np.float32)
        emb_dict[f"id_{id_}"] = stacked

    np.savez_compressed(cache_path, **emb_dict)
    print(f"[{split_name}] Saved → {cache_path}")
    # ⚠ dict() is REQUIRED here too
    return dict(np.load(cache_path, allow_pickle=False))


_train_ids_order = sorted(train_id_to_item.keys())
train_emb_data = generate_embeddings_for_split(
    train_id_to_item, _train_ids_order, "TRAIN", TRAIN_EMB_PATH)

_val_ids_order = sorted(val_id_to_item.keys())
val_emb_data = generate_embeddings_for_split(
    val_id_to_item, _val_ids_order, "VAL", VAL_EMB_PATH)

_test_ids_order = sorted(test_id_to_item.keys())
test_emb_data = generate_embeddings_for_split(
    test_id_to_item, _test_ids_order, "TEST", TEST_EMB_PATH)

print("\nFreeing embedding model memory …")
del _emb_model, _emb_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared.")

[TRAIN] Cache found — loading from /kaggle/working/train_qwen3_embeddings.npz
[VAL] Cache found — loading from /kaggle/working/val_qwen3_embeddings.npz
[TEST] Cache found — loading from /kaggle/working/test_qwen3_embeddings.npz

Freeing embedding model memory …
GPU cache cleared.


## 6. Embedding Verification

In [10]:
def verify_embeddings(emb_data, split_name: str) -> int:
    keys         = list(emb_data.keys())
    dims         = np.array([emb_data[k].shape[1] for k in keys])
    trace_counts = np.array([emb_data[k].shape[0] - 1 for k in keys])
    print(f"\n[{split_name}]  items={len(keys)}")
    print(f"  Embedding dim   : {dims[0]}  uniform={dims.min()==dims.max()}")
    print(f"  Trace counts    : min={trace_counts.min()}  max={trace_counts.max()}  mean={trace_counts.mean():.2f}")
    assert dims.min() == dims.max()
    sample_arr = emb_data[keys[0]]
    print(f"  Sample shape={sample_arr.shape}  claim_norm={np.linalg.norm(sample_arr[0]):.4f}")
    return int(dims[0])

# Ensure plain dict (defensive — in case loaded from cache before this cell)
if not isinstance(train_emb_data, dict): train_emb_data = dict(train_emb_data)
if not isinstance(val_emb_data,   dict): val_emb_data   = dict(val_emb_data)
if not isinstance(test_emb_data,  dict): test_emb_data  = dict(test_emb_data)

d_tr = verify_embeddings(train_emb_data, "TRAIN")
d_va = verify_embeddings(val_emb_data,   "VAL")
d_te = verify_embeddings(test_emb_data,  "TEST")

assert d_tr == d_va == d_te
if d_tr != EMB_DIM:
    EMB_DIM      = d_tr
    COMBINED_DIM = EMB_DIM * 3 + SCALAR_DIM

print(f"\n✓ Embedding dim = {EMB_DIM}")
print(f"✓ COMBINED_DIM  = {COMBINED_DIM}")


[TRAIN]  items=6400
  Embedding dim   : 4096  uniform=True
  Trace counts    : min=1  max=1  mean=1.00
  Sample shape=(2, 4096)  claim_norm=1.0000

[VAL]  items=1600
  Embedding dim   : 4096  uniform=True
  Trace counts    : min=1  max=1  mean=1.00
  Sample shape=(2, 4096)  claim_norm=1.0000

[TEST]  items=2558
  Embedding dim   : 4096  uniform=True
  Trace counts    : min=1  max=1  mean=1.00
  Sample shape=(2, 4096)  claim_norm=1.0000

✓ Embedding dim = 4096
✓ COMBINED_DIM  = 12295


## 7. Feature Engineering

### `split_and_pad`
Splits the stacked embedding array `(1+T, D)` into three tensors:
- **`claim_emb`** `(EMB_DIM,)` — row 0 of the array
- **`traces_padded`** `(TRACE_MAX, EMB_DIM)` — rows 1..T, zero-padded to fixed length
- **`mask`** `(TRACE_MAX,)` — 1.0 for real traces, 0.0 for padding slots

*Preserved exactly from the original notebook.*

### `verdict_features`
Computes **7 scalar features** from the per-trace `Verdict_list` vote distribution:

| Index | Name | Description |
|:---|:---|:---|
| 0 | `frac_false` | Fraction of traces voting "False" |
| 1 | `frac_true` | Fraction of traces voting "True" |
| 2 | `frac_conflicting` | Fraction of traces voting "Conflicting" |
| 3 | `majority_frac` | Fraction held by the plurality class |
| 4 | `distinct_ratio` | Fraction of 3 classes receiving ≥1 vote |
| 5 | `entropy` | Shannon entropy of the vote distribution |
| 6 | `num_traces_norm` | Normalised count of valid traces |

*Preserved exactly from the original notebook.*


In [11]:
def split_and_pad(emb_array: np.ndarray, max_traces=None, emb_dim=None):
    """Row 0=claim, Rows 1+=traces. Zero-pad traces to TRACE_MAX."""
    if max_traces is None: max_traces = TRACE_MAX
    if emb_dim    is None: emb_dim    = EMB_DIM

    claim_emb = emb_array[0].astype(np.float32)
    traces    = emb_array[1:].astype(np.float32)
    n         = traces.shape[0]
    mask      = np.zeros(max_traces, dtype=np.float32)

    if n < max_traces:
        pad           = np.zeros((max_traces - n, emb_dim), dtype=np.float32)
        traces_padded = np.vstack([traces, pad])
        mask[:n]      = 1.0
    else:
        traces_padded     = traces[:max_traces]
        mask[:max_traces] = 1.0

    return claim_emb, traces_padded, mask


def verdict_features(verdict_list: list, max_traces=None) -> np.ndarray:
    """7 scalar features from per-trace vote distribution."""
    if max_traces is None: max_traces = TRACE_MAX

    counts = {0: 0, 1: 0, 2: 0}
    for v in verdict_list:
        if v in label_map:
            counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0:
        return np.zeros(SCALAR_DIM, dtype=np.float32)

    frac_false      = counts[0] / total
    frac_true       = counts[1] / total
    frac_conf       = counts[2] / total
    majority_frac   = max(counts.values()) / total
    distinct_ratio  = sum(1 for c in counts.values() if c > 0) / 3.0
    fracs           = [frac_false, frac_true, frac_conf]
    entropy         = -sum(p * np.log(p + 1e-9) for p in fracs if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)

    return np.array(
        [frac_false, frac_true, frac_conf,
         majority_frac, distinct_ratio, entropy, num_traces_norm],
        dtype=np.float32,
    )

print(f"✓ split_and_pad    → claim=({EMB_DIM},) traces=({TRACE_MAX},{EMB_DIM}) mask=({TRACE_MAX},)")
print(f"✓ verdict_features → ({SCALAR_DIM},)")

✓ split_and_pad    → claim=(4096,) traces=(20,4096) mask=(20,)
✓ verdict_features → (7,)


## 8. Baseline Attention-MLP Architecture

```
Claim Emb (D) ──────────────────────────────────────────────────┐
                                                                │
Traces (T×D) ──► AttentionPool ──► Pooled Emb (D)              │
             ╲──► masked_std   ──► Disagreement (D)     cat ──► BN ──► MLP ──► Logits(3)
                                                                │
Verdict_list ──► verdict_features ──► Scalars (7) ─────────────┘
```

### `AttentionPool` *(preserved exactly)*
Additive (Bahdanau-style) attention with padding mask:
`scores = Linear(Tanh(Linear(trace))) → masked_softmax → weighted_sum`

### `masked_std` *(preserved exactly)*
Element-wise standard deviation over valid (non-padded) trace embeddings.  
Captures *disagreement*: high variance when traces reach different conclusions.

### `TraceMLP` *(baseline — preserved exactly from original notebook)*
`BatchNorm1d(D_combined) → [Linear→BN→ReLU→Dropout(0.3)]×3 → Linear(3)`

### `ConfigurableTraceMLP` *(new — for ablation + pooling comparison)*
Same MLP block; feature groups toggled via `exp_config` dict.  
`combined_dim` is **auto-computed** from which groups are active.

| `pooling_mode` | Method |
|:---|:---|
| `"attention"` | Learned additive attention with masking *(default/baseline)* |
| `"mean"` | Uniform mean over valid traces |
| `"max"` | Element-wise max over valid traces |


In [12]:
# # ============================================================
# # ATTENTION POOLING  (preserved exactly from original notebook)
# # ============================================================

# class AttentionPool(nn.Module):
#     """Additive attention over T trace embeddings with padding mask.

#     Input  : traces (B, T, D),  mask (B, T)  — 1=valid, 0=pad
#     Output : pooled  (B, D)
#     """
#     def __init__(self, dim: int, hidden: int = ATTN_HIDDEN):
#         super().__init__()
#         self.attn = nn.Sequential(
#             nn.Linear(dim, hidden),
#             nn.Tanh(),
#             nn.Linear(hidden, 1),
#         )

#     def forward(self, traces: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
#         scores  = self.attn(traces).squeeze(-1)              # (B, T)
#         scores  = scores.masked_fill(mask == 0, float("-inf"))
#         weights = torch.softmax(scores, dim=1)               # (B, T)
#         pooled  = torch.bmm(weights.unsqueeze(1), traces).squeeze(1)  # (B, D)
#         return pooled


# # ============================================================
# # ALTERNATIVE POOLING FUNCTIONS  (for attention comparison)
# # ============================================================

# def mean_pool(traces: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
#     """Uniform mean over valid (non-padded) trace embeddings."""
#     mask_f = mask.unsqueeze(-1)                        # (B, T, 1)
#     count  = mask_f.sum(dim=1).clamp(min=1.0)         # (B, 1)
#     return (traces * mask_f).sum(dim=1) / count        # (B, D)


# def max_pool(traces: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
#     """Element-wise max over valid trace embeddings (padding → -inf before max)."""
#     mask_exp = mask.unsqueeze(-1).expand_as(traces)
#     traces_m = traces.masked_fill(mask_exp == 0, float("-inf"))
#     result   = traces_m.max(dim=1).values              # (B, D)
#     # Replace any residual -inf (all-masked row, should not occur) with 0
#     return torch.nan_to_num(result, nan=0.0, neginf=0.0)


# # ============================================================
# # DISAGREEMENT VECTOR  (preserved exactly from original notebook)
# # ============================================================

# def masked_std(traces: torch.Tensor,
#                mask:   torch.Tensor,
#                eps:    float = 1e-6) -> torch.Tensor:
#     """Element-wise std-dev across valid trace embeddings (disagreement signal).

#     Input  : traces (B, T, D),  mask (B, T)
#     Output : std    (B, D)
#     """
#     mask_f = mask.unsqueeze(-1)                                       # (B, T, 1)
#     count  = mask_f.sum(dim=1).clamp(min=1.0)                        # (B, 1)
#     mean   = (traces * mask_f).sum(dim=1) / count                    # (B, D)
#     var    = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
#     return torch.sqrt(var + eps)                                      # (B, D)


# # ============================================================
# # BASELINE MODEL  (preserved exactly from original notebook)
# # ============================================================

# class TraceMLP(nn.Module):
#     """Baseline Attention-MLP classifier — original notebook implementation.

#     Concatenation order: [claim | attn_pooled | disagreement | scalars]
#     Combined dim       : EMB_DIM*3 + SCALAR_DIM
#     """
#     def __init__(self,
#                  combined_dim: int  = None,
#                  hidden_dims:  list = None,
#                  output_dim:   int  = OUTPUT_DIM):
#         super().__init__()
#         if combined_dim is None: combined_dim = COMBINED_DIM
#         if hidden_dims  is None: hidden_dims  = HIDDEN_DIMS

#         self.attn_pool = AttentionPool(EMB_DIM)

#         layers = [nn.BatchNorm1d(combined_dim)]
#         prev   = combined_dim
#         for h in hidden_dims:
#             layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
#                        nn.ReLU(), nn.Dropout(0.3)]
#             prev = h
#         layers.append(nn.Linear(prev, output_dim))
#         self.mlp = nn.Sequential(*layers)

#     def forward(self, claim, traces, mask, scalars):
#         pooled       = self.attn_pool(traces, mask)   # (B, D)
#         disagreement = masked_std(traces, mask)        # (B, D)
#         x = torch.cat([claim, pooled, disagreement, scalars], dim=1)
#         return self.mlp(x)                             # (B, 3)


# # ============================================================
# # CONFIGURABLE MODEL  (new — ablation + pooling comparison)
# # ============================================================

# class ConfigurableTraceMLP(nn.Module):
#     """Configurable Attention-MLP for ablation study and pooling comparison.

#     exp_config keys
#     ---------------
#     use_claim        : bool — include claim embedding in input
#     use_pool         : bool — include a pooled trace representation
#     use_disagreement : bool — include the masked_std disagreement vector
#     use_verdict      : bool — include 7-dim verdict scalar features
#     pooling_mode     : str  — "attention" | "mean" | "max"

#     combined_dim is computed automatically; MLP architecture is identical to
#     TraceMLP (preserved from original).
#     """
#     def __init__(self,
#                  exp_config:  dict,
#                  emb_dim:     int  = None,
#                  scalar_dim:  int  = None,
#                  hidden_dims: list = None,
#                  output_dim:  int  = OUTPUT_DIM):
#         super().__init__()
#         if emb_dim    is None: emb_dim    = EMB_DIM
#         if scalar_dim is None: scalar_dim = SCALAR_DIM
#         if hidden_dims is None: hidden_dims = HIDDEN_DIMS

#         self.use_claim    = exp_config.get("use_claim",        True)
#         self.use_pool     = exp_config.get("use_pool",         True)
#         self.use_disag    = exp_config.get("use_disagreement", True)
#         self.use_verdict  = exp_config.get("use_verdict",      True)
#         self.pooling_mode = exp_config.get("pooling_mode",     "attention")
#         self.emb_dim      = emb_dim
#         self.scalar_dim   = scalar_dim

#         if self.use_pool and self.pooling_mode == "attention":
#             self.attn_pool = AttentionPool(emb_dim)

#         # Auto-compute combined input dimension
#         combined_dim = 0
#         if self.use_claim:   combined_dim += emb_dim
#         if self.use_pool:    combined_dim += emb_dim
#         if self.use_disag:   combined_dim += emb_dim
#         if self.use_verdict: combined_dim += scalar_dim
#         assert combined_dim > 0, "At least one feature group must be enabled."
#         self.combined_dim = combined_dim

#         layers = [nn.BatchNorm1d(combined_dim)]
#         prev   = combined_dim
#         for h in hidden_dims:
#             layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
#                        nn.ReLU(), nn.Dropout(0.3)]
#             prev = h
#         layers.append(nn.Linear(prev, output_dim))
#         self.mlp = nn.Sequential(*layers)

#     def forward(self, claim, traces, mask, scalars):
#         parts = []

#         if self.use_pool:
#             if   self.pooling_mode == "attention":
#                 pooled = self.attn_pool(traces, mask)
#             elif self.pooling_mode == "mean":
#                 pooled = mean_pool(traces, mask)
#             elif self.pooling_mode == "max":
#                 pooled = max_pool(traces, mask)
#             else:
#                 raise ValueError(f"Unknown pooling_mode: {self.pooling_mode!r}")

#         # Canonical concatenation order (matches TraceMLP baseline)
#         if self.use_claim:   parts.append(claim)
#         if self.use_pool:    parts.append(pooled)
#         if self.use_disag:   parts.append(masked_std(traces, mask))
#         if self.use_verdict: parts.append(scalars)

#         return self.mlp(torch.cat(parts, dim=1))


# # ── Shape sanity checks ──────────────────────────────────────
# _B, _T, _D = 2, TRACE_MAX, EMB_DIM
# _c, _tr, _mk, _sc = (torch.randn(_B, _D), torch.randn(_B, _T, _D),
#                       torch.ones(_B, _T),  torch.randn(_B, SCALAR_DIM))

# with torch.no_grad():
#     _base = TraceMLP()
#     assert _base(_c, _tr, _mk, _sc).shape == (_B, OUTPUT_DIM)
#     print(f"✓ TraceMLP (baseline)       combined_dim={COMBINED_DIM}"
#           f"  output={(_B, OUTPUT_DIM)}")

#     for _name, _cfg in [
#         ("A_Full",        dict(use_claim=True, use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
#         ("B_NoClaim",     dict(use_claim=False,use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
#         ("C_NoAttention", dict(use_claim=True, use_pool=False, use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
#         ("D_NoDisag",     dict(use_claim=True, use_pool=True,  use_disagreement=False, use_verdict=True,  pooling_mode="attention")),
#         ("E_NoVerdict",   dict(use_claim=True, use_pool=True,  use_disagreement=True,  use_verdict=False, pooling_mode="attention")),
#         ("Mean",          dict(use_claim=True, use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="mean")),
#         ("Max",           dict(use_claim=True, use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="max")),
#     ]:
#         _m = ConfigurableTraceMLP(_cfg)
#         _o = _m(_c, _tr, _mk, _sc)
#         assert _o.shape == (_B, OUTPUT_DIM)
#         print(f"✓ ConfigurableTraceMLP [{_name:<14}] combined_dim={_m.combined_dim}")

# del _base, _m, _c, _tr, _mk, _sc, _o


# Model Architecture

In [13]:
# ── Attention Pooling (preserved from original) ───────────────────────────────
class AttentionPool(nn.Module):
    def __init__(self, dim: int, hidden: int = ATTN_HIDDEN):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1),
        )
    def forward(self, traces, mask):
        scores  = self.attn(traces).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return torch.bmm(weights.unsqueeze(1), traces).squeeze(1)

# ── Alternative Pooling (for attention comparison) ────────────────────────────
def mean_pool(traces, mask):
    mask_f = mask.unsqueeze(-1)
    count  = mask_f.sum(dim=1).clamp(min=1.0)
    return (traces * mask_f).sum(dim=1) / count

def max_pool(traces, mask):
    mask_exp = mask.unsqueeze(-1).expand_as(traces)
    result   = traces.masked_fill(mask_exp == 0, float("-inf")).max(dim=1).values
    return torch.nan_to_num(result, nan=0.0, neginf=0.0)

# ── Disagreement Vector (preserved from original) ─────────────────────────────
def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)
    count  = mask_f.sum(dim=1).clamp(min=1.0)
    mean   = (traces * mask_f).sum(dim=1) / count
    var    = ((traces - mean.unsqueeze(1)) ** 2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)

# ── Baseline TraceMLP (preserved from original) ───────────────────────────────
class TraceMLP(nn.Module):
    def __init__(self, combined_dim=None, hidden_dims=None, output_dim=OUTPUT_DIM):
        super().__init__()
        if combined_dim is None: combined_dim = COMBINED_DIM
        if hidden_dims  is None: hidden_dims  = HIDDEN_DIMS
        self.attn_pool = AttentionPool(EMB_DIM)
        layers = [nn.BatchNorm1d(combined_dim)]
        prev   = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        pooled       = self.attn_pool(traces, mask)
        disagreement = masked_std(traces, mask)
        return self.mlp(torch.cat([claim, pooled, disagreement, scalars], dim=1))

# ── Configurable Model (ablation + pooling comparison) ────────────────────────
class ConfigurableTraceMLP(nn.Module):
    """
    exp_config keys:
        use_claim, use_pool, use_disagreement, use_verdict : bool
        pooling_mode : "attention" | "mean" | "max"
    combined_dim is auto-computed.
    """
    def __init__(self, exp_config, emb_dim=None, scalar_dim=None,
                 hidden_dims=None, output_dim=OUTPUT_DIM):
        super().__init__()
        if emb_dim    is None: emb_dim    = EMB_DIM
        if scalar_dim is None: scalar_dim = SCALAR_DIM
        if hidden_dims is None: hidden_dims = HIDDEN_DIMS

        self.use_claim    = exp_config.get("use_claim",        True)
        self.use_pool     = exp_config.get("use_pool",         True)
        self.use_disag    = exp_config.get("use_disagreement", True)
        self.use_verdict  = exp_config.get("use_verdict",      True)
        self.pooling_mode = exp_config.get("pooling_mode",     "attention")
        self.emb_dim      = emb_dim
        self.scalar_dim   = scalar_dim

        if self.use_pool and self.pooling_mode == "attention":
            self.attn_pool = AttentionPool(emb_dim)

        combined_dim = 0
        if self.use_claim:   combined_dim += emb_dim
        if self.use_pool:    combined_dim += emb_dim
        if self.use_disag:   combined_dim += emb_dim
        if self.use_verdict: combined_dim += scalar_dim
        assert combined_dim > 0
        self.combined_dim = combined_dim

        layers = [nn.BatchNorm1d(combined_dim)]
        prev   = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        parts = []
        if self.use_pool:
            if   self.pooling_mode == "attention": pooled = self.attn_pool(traces, mask)
            elif self.pooling_mode == "mean":      pooled = mean_pool(traces, mask)
            elif self.pooling_mode == "max":       pooled = max_pool(traces, mask)
            else: raise ValueError(f"Unknown pooling_mode: {self.pooling_mode!r}")

        if self.use_claim:   parts.append(claim)
        if self.use_pool:    parts.append(pooled)
        if self.use_disag:   parts.append(masked_std(traces, mask))
        if self.use_verdict: parts.append(scalars)
        return self.mlp(torch.cat(parts, dim=1))

# ── Quick shape check ─────────────────────────────────────────────────────────
_B, _T, _D = 2, TRACE_MAX, EMB_DIM
_c, _tr, _mk, _sc = (torch.randn(_B,_D), torch.randn(_B,_T,_D),
                      torch.ones(_B,_T),  torch.randn(_B,SCALAR_DIM))
with torch.no_grad():
    assert TraceMLP()(_c,_tr,_mk,_sc).shape == (_B, OUTPUT_DIM)
    for _n, _cfg in [
        ("A_Full",       dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
        ("B_NoClaim",    dict(use_claim=False, use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
        ("C_NoAttn",     dict(use_claim=True,  use_pool=False, use_disagreement=True,  use_verdict=True,  pooling_mode="attention")),
        ("D_NoDisag",    dict(use_claim=True,  use_pool=True,  use_disagreement=False, use_verdict=True,  pooling_mode="attention")),
        ("E_NoVerdict",  dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=False, pooling_mode="attention")),
        ("Mean",         dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="mean")),
        ("Max",          dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="max")),
    ]:
        _m = ConfigurableTraceMLP(_cfg)
        assert _m(_c,_tr,_mk,_sc).shape == (_B, OUTPUT_DIM)
        print(f"✓ [{_n:<14}] combined_dim={_m.combined_dim}")
del _c,_tr,_mk,_sc,_m

✓ [A_Full        ] combined_dim=12295
✓ [B_NoClaim     ] combined_dim=8199
✓ [C_NoAttn      ] combined_dim=8199
✓ [D_NoDisag     ] combined_dim=8199
✓ [E_NoVerdict   ] combined_dim=12288
✓ [Mean          ] combined_dim=12295
✓ [Max           ] combined_dim=12295


## 9. Training Pipeline

In [14]:
# ============================================================
# PYTORCH DATASET CLASSES  (preserved exactly from original)
# ============================================================

class BaseDataset(Dataset):
    label_key = "label"
    def __getitem__(self, idx):
        id_       = self.ids[idx]
        emb_array = self.emb_data[f"id_{id_}"]
        claim_emb, traces_padded, mask = split_and_pad(emb_array)
        item         = self.id_to_item[id_]
        label        = label_map[item[self.label_key]]
        verdict_list = item.get(VERDICT_LIST_FIELD, [])
        scalars      = verdict_features(verdict_list)
        return (
            torch.from_numpy(claim_emb),
            torch.from_numpy(traces_padded),
            torch.from_numpy(mask),
            torch.from_numpy(scalars),
            torch.tensor(label, dtype=torch.long),
        )
    def __len__(self): return len(self.ids)

class TrainDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data   = train_emb_data
        self.id_to_item = train_id_to_item
        all_ids  = sorted(int(k.split("_")[1]) for k in self.emb_data.keys() if k.startswith("id_"))
        self.ids = [i for i in all_ids if i in self.id_to_item]
        print(f"TRAIN dataset : {len(self.ids)} samples")

class ValDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data   = val_emb_data
        self.id_to_item = val_id_to_item
        self.ids        = sorted(self.id_to_item.keys())
        print(f"VAL   dataset : {len(self.ids)} samples")

class TestDataset(BaseDataset):
    label_key = "Label"   # capital L in test set
    def __init__(self):
        self.emb_data   = test_emb_data
        self.id_to_item = test_id_to_item
        self.ids        = sorted(self.id_to_item.keys())
        print(f"TEST  dataset : {len(self.ids)} samples")

set_seed()
train_dataset = TrainDataset()
val_dataset   = ValDataset()
test_dataset  = TestDataset()

_train_label_list = [label_map[train_dataset.id_to_item[i]["label"]] for i in train_dataset.ids]
_label_counts = Counter(_train_label_list)
_total        = sum(_label_counts.values())
class_weights = torch.tensor(
    [_total / (3.0 * _label_counts[c]) for c in range(3)], dtype=torch.float32
).to(DEVICE)

print(f"\nLabel distribution : {dict(_label_counts)}")
print(f"Class weights      : {[round(w, 4) for w in class_weights.tolist()]}")

# ⚠ num_workers=0 — NpzFile/dict in forked workers causes zlib errors on Kaggle
_dl_kw = dict(num_workers=0, pin_memory=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_dl_kw)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

TRAIN dataset : 6400 samples
VAL   dataset : 1600 samples
TEST  dataset : 2558 samples

Label distribution : {2: 1508, 0: 3717, 1: 1175}
Class weights      : [0.5739, 1.8156, 1.4147]

Train batches : 400
Val batches   : 100
Test batches  : 160


## 10. Evaluation Pipeline

In [15]:
def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss = 0.0
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True),   traces.to(device, non_blocking=True),
            mask.to(device,  non_blocking=True),   scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True),
        )
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                logits = model(claim, traces, mask, scalars)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(claim, traces, mask, scalars)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True),   traces.to(device, non_blocking=True),
            mask.to(device,  non_blocking=True),   scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True),
        )
        logits = model(claim, traces, mask, scalars)
        total_loss += criterion(logits, labels).item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    return total_loss / len(loader), acc, all_preds, all_labels


def generate_report(all_labels, all_preds, set_name="VAL", save_prefix=None):
    all_labels = np.array(all_labels, dtype=int)
    all_preds  = np.array(all_preds,  dtype=int)
    cm  = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2])
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n{'='*70}\n{set_name} SET — METRICS\n{'='*70}")
    print("Confusion Matrix (rows=true, cols=pred | False/True/Conflicting):")
    print(cm, "\n")

    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall    = recall_score(   all_labels, all_preds, average=None, zero_division=0)
    f1        = f1_score(       all_labels, all_preds, average=None, zero_division=0)

    print(f"{'Class':<10}{'Label':<16}{'Precision':<12}{'Recall':<12}{'F1':<12}")
    print("-"*62)
    for c in range(3):
        print(f"{c:<10}{reverse_label_map[c]:<16}{precision[c]:<12.4f}{recall[c]:<12.4f}{f1[c]:<12.4f}")
    print("-"*62)
    macro_p, macro_r, macro_f1 = float(np.mean(precision)), float(np.mean(recall)), float(np.mean(f1))
    print(f"{'MACRO AVG':<26}{macro_p:<12.4f}{macro_r:<12.4f}{macro_f1:<12.4f}\n")

    wp  = float(precision_score(all_labels, all_preds, average="weighted", zero_division=0))
    wr  = float(recall_score(   all_labels, all_preds, average="weighted", zero_division=0))
    wf1 = float(f1_score(       all_labels, all_preds, average="weighted", zero_division=0))
    print(f"Accuracy: {acc:.4f}  |  Weighted F1: {wf1:.4f}")

    pfx = save_prefix if save_prefix else set_name.lower().replace(" ", "_")
    try:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=list(reverse_label_map.values()),
                    yticklabels=list(reverse_label_map.values()))
        ax.set_title(f"{set_name} — Confusion Matrix")
        ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{pfx}_confusion_matrix.png", dpi=100, bbox_inches="tight")
        plt.close()
    except Exception as exc:
        print(f"Warning: plot failed: {exc}")

    return {
        "accuracy": float(acc), "cm": cm.tolist(),
        "precision": numpy_to_python(precision), "recall": numpy_to_python(recall),
        "f1": numpy_to_python(f1),
        "avg_precision": macro_p, "avg_recall": macro_r, "avg_f1": macro_f1,
        "macro_f1": macro_f1,
        "weighted_precision": wp, "weighted_recall": wr, "weighted_f1": wf1,
    }


def run_experiment(exp_name, exp_config, train_loader, val_loader, test_loader,
                   weights, device=DEVICE, use_amp=True):
    print(f"\n{'#'*72}\n# EXPERIMENT : {exp_name}\n# Config: {exp_config}\n{'#'*72}")
    set_seed()

    model     = ConfigurableTraceMLP(exp_config).to(device)
    _cdim     = model.combined_dim
    n_params  = sum(p.numel() for p in model.parameters())
    print(f"combined_dim={_cdim}  params={n_params:,}")

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"DataParallel across {torch.cuda.device_count()} GPUs")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler    = GradScaler() if (use_amp and device.startswith("cuda")) else None

    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    train_losses, val_losses = [], []
    t0 = time.time()

    for epoch in range(MAX_ITERS):
        tr_loss             = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        va_loss, va_acc,_,_ = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr_loss); val_losses.append(va_loss)
        scheduler.step(va_loss)
        print(f"  Ep {epoch+1:3d}/{MAX_ITERS} | Train {tr_loss:.4f} | Val {va_loss:.4f} | ValAcc {va_acc:.4f}")

        if va_loss < best_val_loss:
            best_val_loss, patience_counter = va_loss, 0
            _base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = copy.deepcopy(_base.state_dict())
            print("    ✓ Best model saved")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - t0
    _base = model.module if isinstance(model, nn.DataParallel) else model
    _base.load_state_dict(best_state)

    pfx = exp_name.lower().replace(" ", "_")
    tr_loss_f,_,tr_preds,tr_labels = evaluate(model, train_loader, criterion, device)
    va_loss_f,_,va_preds,va_labels = evaluate(model, val_loader,   criterion, device)
    te_loss_f,_,te_preds,te_labels = evaluate(model, test_loader,  criterion, device)

    train_metrics = generate_report(tr_labels, tr_preds, f"{exp_name} TRAIN", f"{pfx}_train")
    val_metrics   = generate_report(va_labels, va_preds, f"{exp_name} VAL",   f"{pfx}_val")
    test_metrics  = generate_report(te_labels, te_preds, f"{exp_name} TEST",  f"{pfx}_test")

    del model; gc.collect(); torch.cuda.empty_cache()

    return {
        "experiment": exp_name, "config": exp_config,
        "train_loss": float(tr_loss_f), "val_loss": float(va_loss_f), "test_loss": float(te_loss_f),
        "train_metrics": train_metrics, "val_metrics": val_metrics, "test_metrics": test_metrics,
        "train_losses": [float(x) for x in train_losses],
        "val_losses":   [float(x) for x in val_losses],
        "training_time_s": float(elapsed),
    }

print("All functions ready.")

All functions ready.


## 11. Ablation Study

### Ablation Configurations (A–E)

| Experiment | `use_claim` | `use_pool` | `use_disagreement` | `use_verdict` | `pooling_mode` |
|:---|:---:|:---:|:---:|:---:|:---:|
| A — Full Model | ✓ | ✓ | ✓ | ✓ | attention |
| B — No Claim | ✗ | ✓ | ✓ | ✓ | attention |
| C — No Attention | ✓ | ✗ | ✓ | ✓ | — |
| D — No Disagreement | ✓ | ✓ | ✗ | ✓ | attention |
| E — No Verdict Features | ✓ | ✓ | ✓ | ✗ | attention |

### Attention Comparison Configurations

| Experiment | `pooling_mode` | Note |
|:---|:---|:---|
| Attn_MeanPool | mean | Uniform average over valid traces |
| Attn_MaxPool | max | Element-wise max over valid traces |

*A_Full (attention) is the attention baseline — already included in A–E.*


In [16]:
ABLATION_CONFIGS = {
    "A_Full":           dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention"),
    "B_NoClaim":        dict(use_claim=False, use_pool=True,  use_disagreement=True,  use_verdict=True,  pooling_mode="attention"),
    "C_NoAttention":    dict(use_claim=True,  use_pool=False, use_disagreement=True,  use_verdict=True,  pooling_mode="attention"),
    "D_NoDisagreement": dict(use_claim=True,  use_pool=True,  use_disagreement=False, use_verdict=True,  pooling_mode="attention"),
    "E_NoVerdict":      dict(use_claim=True,  use_pool=True,  use_disagreement=True,  use_verdict=False, pooling_mode="attention"),
}

ATTENTION_COMPARISON_CONFIGS = {
    "Attn_MeanPool": dict(use_claim=True, use_pool=True, use_disagreement=True, use_verdict=True, pooling_mode="mean"),
    "Attn_MaxPool":  dict(use_claim=True, use_pool=True, use_disagreement=True, use_verdict=True, pooling_mode="max"),
}

print(f"{'Experiment':<22}  {'combined_dim':>14}")
print("-"*40)
for _name, _cfg in {**ABLATION_CONFIGS, **ATTENTION_COMPARISON_CONFIGS}.items():
    _d = (EMB_DIM*_cfg["use_claim"] + EMB_DIM*_cfg["use_pool"] +
          EMB_DIM*_cfg["use_disagreement"] + SCALAR_DIM*_cfg["use_verdict"])
    print(f"{_name:<22}  {_d:>14}")

Experiment                combined_dim
----------------------------------------
A_Full                           12295
B_NoClaim                         8199
C_NoAttention                     8199
D_NoDisagreement                  8199
E_NoVerdict                      12288
Attn_MeanPool                    12295
Attn_MaxPool                     12295


In [17]:
all_results = []
for _exp_name, _exp_cfg in ABLATION_CONFIGS.items():
    _res = run_experiment(
        exp_name=_exp_name, exp_config=_exp_cfg,
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights, device=DEVICE, use_amp=True,
    )
    all_results.append(_res)
print(f"\n✓ Ablation study complete — {len(all_results)} experiments run.")


########################################################################
# EXPERIMENT : A_Full
# Config: {'use_claim': True, 'use_pool': True, 'use_disagreement': True, 'use_verdict': True, 'pooling_mode': 'attention'}
########################################################################
combined_dim=12295  params=7,535,634
DataParallel across 2 GPUs
  Ep   1/100 | Train 0.9991 | Val 0.8119 | ValAcc 0.6181
    ✓ Best model saved
  Ep   2/100 | Train 0.9133 | Val 0.7931 | ValAcc 0.6281
    ✓ Best model saved
  Ep   3/100 | Train 0.8553 | Val 0.7735 | ValAcc 0.6375
    ✓ Best model saved
  Ep   4/100 | Train 0.8081 | Val 0.8077 | ValAcc 0.6344
  Ep   5/100 | Train 0.7607 | Val 0.9614 | ValAcc 0.6206
  Ep   6/100 | Train 0.6979 | Val 1.1699 | ValAcc 0.6100
  Ep   7/100 | Train 0.6639 | Val 1.2014 | ValAcc 0.6088
  Ep   8/100 | Train 0.5503 | Val 1.2905 | ValAcc 0.6169
  Ep   9/100 | Train 0.5072 | Val 2.0129 | ValAcc 0.5800
  Ep  10/100 | Train 0.4575 | Val 2.4910 | ValAcc 0.5737
  Ep

In [26]:
for _exp_name, _exp_cfg in ATTENTION_COMPARISON_CONFIGS.items():
    _res = run_experiment(
        exp_name=_exp_name, exp_config=_exp_cfg,
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights, device=DEVICE, use_amp=True,
    )
    all_results.append(_res)
print(f"\n✓ Attention comparison complete — total experiments: {len(all_results)}")


########################################################################
# EXPERIMENT : Attn_MeanPool
# Config: {'use_claim': True, 'use_pool': True, 'use_disagreement': True, 'use_verdict': True, 'pooling_mode': 'mean'}
########################################################################
combined_dim=12295  params=6,486,545
DataParallel across 2 GPUs
  Ep   1/100 | Train 0.9923 | Val 0.8367 | ValAcc 0.5975
    ✓ Best model saved
  Ep   2/100 | Train 0.9092 | Val 0.8196 | ValAcc 0.6188
    ✓ Best model saved
  Ep   3/100 | Train 0.8559 | Val 0.7596 | ValAcc 0.6469
    ✓ Best model saved
  Ep   4/100 | Train 0.8082 | Val 0.8009 | ValAcc 0.6462
  Ep   5/100 | Train 0.7474 | Val 0.8582 | ValAcc 0.6281
  Ep   6/100 | Train 0.7058 | Val 1.0092 | ValAcc 0.6225
  Ep   7/100 | Train 0.6519 | Val 1.1242 | ValAcc 0.6206
  Ep   8/100 | Train 0.5601 | Val 1.0462 | ValAcc 0.6350
  Ep   9/100 | Train 0.4964 | Val 1.3445 | ValAcc 0.6256
  Ep  10/100 | Train 0.4675 | Val 1.7944 | ValAcc 0.5831
  

## 12. Results

In [28]:
def build_summary_rows(results):
    rows = []
    for r in results:
        rows.append({
            "Experiment":       r["experiment"],
            "Pooling":          r["config"].get("pooling_mode", "-"),
            "Val_Macro_F1":     r["val_metrics"]["macro_f1"],
            "Val_Weighted_F1":  r["val_metrics"]["weighted_f1"],
            "Val_Accuracy":     r["val_metrics"]["accuracy"],
            "Test_Macro_F1":    r["test_metrics"]["macro_f1"],
            "Test_Weighted_F1": r["test_metrics"]["weighted_f1"],
            "Test_Accuracy":    r["test_metrics"]["accuracy"],
            "Train_Loss":       r["train_loss"],
            "Val_Loss":         r["val_loss"],
            "Test_Loss":        r["test_loss"],
            "Training_Time_s":  r["training_time_s"],
        })
    return sorted(rows, key=lambda x: x["Val_Macro_F1"], reverse=True)
table_rows = build_summary_rows(all_results)
print("="*100)
for row in table_rows:
    print(f"{row['Experiment']:<22} {row['Pooling']:<10}"
          f" {row['Val_Macro_F1']:>9.4f} {row['Val_Weighted_F1']:>9.4f} {row['Val_Accuracy']:>8.4f}"
          f" {row['Test_Macro_F1']:>9.4f} {row['Test_Weighted_F1']:>9.4f} {row['Test_Accuracy']:>8.4f}")

print("="*100)

_csv_path = f"{OUTPUT_DIR}/ablation_results.csv"
with open(_csv_path, "w", newline="") as _f:
    writer = csv.DictWriter(_f, fieldnames=list(table_rows[0].keys()))
    writer.writeheader(); writer.writerows(table_rows)
print(f"\nCSV  saved → {_csv_path}")

_json_path = f"{OUTPUT_DIR}/ablation_results_full.json"
with open(_json_path, "w") as _f:
    json.dump(numpy_to_python(all_results), _f, indent=2)
print(f"JSON saved → {_json_path}")

Attn_MeanPool          mean          0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MaxPool           max           0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MeanPool          mean          0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MaxPool           max           0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
E_NoVerdict            attention     0.5587    0.6296   0.6162    0.4372    0.6209   0.6083
A_Full                 attention     0.5558    0.6307   0.6375    0.5075    0.6755   0.6548
C_NoAttention          attention     0.5557    0.6352   0.6450    0.5184    0.6898   0.6806
D_NoDisagreement       attention     0.5309    0.6184   0.6406    0.5152    0.6815   0.6724
B_NoClaim              attention     0.2422    0.4146   0.5706    0.5948    0.7721   0.7643

CSV  saved → /kaggle/working/ablation_results.csv
JSON saved → /kaggle/working/ablation_results_full.json


# MACRO AVG AND FINAL RESULTs

In [29]:
# ============================================================
# Summary Table: Macro F1, Macro Recall and Average
# ============================================================

def build_macro_summary(results):
    rows = []

    for r in results:
        val_f1 = float(r["val_metrics"]["macro_f1"])
        val_rec = float(r["val_metrics"]["avg_recall"])

        test_f1 = float(r["test_metrics"]["macro_f1"])
        test_rec = float(r["test_metrics"]["avg_recall"])

        rows.append({
            "Experiment": r["experiment"],
            "Pooling": r["config"].get("pooling_mode", "-"),

            "Val_Macro_F1": val_f1,
            "Val_Macro_Recall": val_rec,
            "Val_Avg": (val_f1 + val_rec) / 2,

            "Test_Macro_F1": test_f1,
            "Test_Macro_Recall": test_rec,
            "Test_Avg": (test_f1 + test_rec) / 2,
        })

    return sorted(rows, key=lambda x: x["Val_Avg"], reverse=True)


table_rows = build_macro_summary(all_results)

print("\n" + "=" * 100)
print("FINAL RESULTS TABLE (Sorted by Validation Average)")
print("=" * 100)

print(
    f"{'Experiment':<25}"
    f"{'Pooling':<12}"
    f"{'ValF1':>10}"
    f"{'ValRec':>10}"
    f"{'ValAvg':>10}"
    f"{'TestF1':>10}"
    f"{'TestRec':>10}"
    f"{'TestAvg':>10}"
)

print("-" * 100)

for row in table_rows:
    print(
        f"{row['Experiment']:<25}"
        f"{row['Pooling']:<12}"
        f"{row['Val_Macro_F1']:>10.4f}"
        f"{row['Val_Macro_Recall']:>10.4f}"
        f"{row['Val_Avg']:>10.4f}"
        f"{row['Test_Macro_F1']:>10.4f}"
        f"{row['Test_Macro_Recall']:>10.4f}"
        f"{row['Test_Avg']:>10.4f}"
    )

print("=" * 100)


# ---------------- Save CSV ----------------

csv_path = f"{OUTPUT_DIR}/macro_f1_recall_summary.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)

print(f"\nCSV saved -> {csv_path}")


# ---------------- Save JSON ----------------

json_path = f"{OUTPUT_DIR}/macro_f1_recall_summary.json"

with open(json_path, "w") as f:
    json.dump(numpy_to_python(table_rows), f, indent=2)

print(f"JSON saved -> {json_path}")


FINAL RESULTS TABLE (Sorted by Validation Average)
Experiment               Pooling          ValF1    ValRec    ValAvg    TestF1   TestRec   TestAvg
----------------------------------------------------------------------------------------------------
E_NoVerdict              attention       0.5587    0.5819    0.5703    0.4372    0.4435    0.4404
Attn_MeanPool            mean            0.5649    0.5670    0.5659    0.4542    0.4719    0.4630
Attn_MaxPool             max             0.5649    0.5670    0.5659    0.4542    0.4719    0.4630
Attn_MeanPool            mean            0.5649    0.5670    0.5659    0.4542    0.4719    0.4630
Attn_MaxPool             max             0.5649    0.5670    0.5659    0.4542    0.4719    0.4630
C_NoAttention            attention       0.5557    0.5537    0.5547    0.5184    0.5306    0.5245
A_Full                   attention       0.5558    0.5509    0.5533    0.5075    0.5205    0.5140
D_NoDisagreement         attention       0.5309    0.5423    0.

# Plots

In [30]:
# ── Bar chart ─────────────────────────────────────────────────────────────────
_ablation_names = list(ABLATION_CONFIGS.keys())
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, key, title in [(axes[0],"Val_Macro_F1","Validation Macro F1"),(axes[1],"Test_Macro_F1","Test Macro F1")]:
    scores  = [r[key] for r in table_rows]
    names   = [r["Experiment"] for r in table_rows]
    colours = ["#1565C0" if n in _ablation_names else "#E65100" for n in names]
    bars = ax.barh(names, scores, color=colours, edgecolor="white", linewidth=0.5)
    ax.set_xlim(0, 1.0); ax.set_xlabel("Macro F1"); ax.set_title(title, fontweight="bold")
    ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=9); ax.invert_yaxis()
    ax.axvline(max(scores), color="red", linestyle="--", linewidth=1.2, alpha=0.7, label=f"Best: {max(scores):.4f}")
    ax.legend(fontsize=9); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

fig.legend(handles=[Patch(facecolor="#1565C0", label="Ablation (A–E)"),
                    Patch(facecolor="#E65100", label="Attention comparison")],
           loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5,-0.05))
plt.suptitle("Ablation Study + Attention Comparison\nCLEF CheckThat! 2026 Task 2  —  Qwen3-Embedding-8B",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ablation_macro_f1_chart.png", dpi=120, bbox_inches="tight"); plt.close()
print(f"Bar chart saved → {OUTPUT_DIR}/ablation_macro_f1_chart.png")

# ── Loss curves ───────────────────────────────────────────────────────────────
_n = len(all_results); _cols = 4; _rows_p = (_n+_cols-1)//_cols
fig, axes = plt.subplots(_rows_p, _cols, figsize=(_cols*4, _rows_p*3.5))
axes_flat = axes.flatten() if hasattr(axes,"flatten") else [axes]
for i, r in enumerate(all_results):
    ax = axes_flat[i]
    eps = range(1, len(r["train_losses"])+1)
    ax.plot(eps, r["train_losses"], label="Train", linewidth=1.5, color="#1565C0")
    ax.plot(eps, r["val_losses"],   label="Val",   linewidth=1.5, color="#E53935")
    ax.set_title(r["experiment"], fontsize=9, fontweight="bold")
    ax.set_xlabel("Epoch",fontsize=8); ax.set_ylabel("Loss",fontsize=8); ax.legend(fontsize=7)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
plt.suptitle("Train / Validation Loss Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_curves.png", dpi=100, bbox_inches="tight"); plt.close()
print(f"Loss curves saved → {OUTPUT_DIR}/loss_curves.png")

print("\n── Output files ───────────────────────────────────────────────────────")
for _fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {_fn:<60}  {os.path.getsize(os.path.join(OUTPUT_DIR,_fn))/1e3:>10.1f} KB")

Bar chart saved → /kaggle/working/ablation_macro_f1_chart.png
Loss curves saved → /kaggle/working/loss_curves.png

── Output files ───────────────────────────────────────────────────────
  .virtual_documents                                                   4.1 KB
  a_full_test_confusion_matrix.png                                    27.9 KB
  a_full_train_confusion_matrix.png                                   29.0 KB
  a_full_val_confusion_matrix.png                                     27.8 KB
  ablation_macro_f1_chart.png                                         78.0 KB
  ablation_results.csv                                                 2.0 KB
  ablation_results_full.json                                          38.6 KB
  attn_maxpool_test_confusion_matrix.png                              29.5 KB
  attn_maxpool_train_confusion_matrix.png                             29.7 KB
  attn_maxpool_val_confusion_matrix.png                               28.9 KB
  attn_meanpool_test_confusion_ma

In [ ]:
# # ============================================================
# # VISUALISATION — Training / Validation loss curves per experiment
# # ============================================================

# _n = len(all_results)
# _cols = 4
# _rows_plot = (_n + _cols - 1) // _cols

# fig, axes = plt.subplots(_rows_plot, _cols,
#                          figsize=(_cols * 4, _rows_plot * 3.5))
# axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

# for i, r in enumerate(all_results):
#     ax  = axes_flat[i]
#     eps = range(1, len(r["train_losses"]) + 1)
#     ax.plot(eps, r["train_losses"], label="Train", linewidth=1.5, color="#1565C0")
#     ax.plot(eps, r["val_losses"],   label="Val",   linewidth=1.5, color="#E53935")
#     ax.set_title(r["experiment"], fontsize=9, fontweight="bold")
#     ax.set_xlabel("Epoch", fontsize=8)
#     ax.set_ylabel("Loss",  fontsize=8)
#     ax.legend(fontsize=7)
#     ax.spines["top"].set_visible(False)
#     ax.spines["right"].set_visible(False)

# for j in range(i + 1, len(axes_flat)):
#     axes_flat[j].set_visible(False)

# plt.suptitle("Train / Validation Loss Curves per Experiment",
#              fontsize=13, fontweight="bold")
# plt.tight_layout()
# _lc_path = f"{OUTPUT_DIR}/loss_curves.png"
# plt.savefig(_lc_path, dpi=100, bbox_inches="tight")
# plt.close()
# print(f"Loss curves saved → {_lc_path}")

# # ── Final file manifest ────────────────────────────────────────
# print("\n── Output files in", OUTPUT_DIR, "──────────────────────────────────")
# for _fn in sorted(os.listdir(OUTPUT_DIR)):
#     _fp   = os.path.join(OUTPUT_DIR, _fn)
#     _size = os.path.getsize(_fp)
#     print(f"  {_fn:<60}  {_size/1e3:>10.1f} KB")


## 13. Discussion

### Interpreting the Ablation Results

**Experiment A — Full Model (Baseline)**  
All four feature groups active: claim + attention-pooled traces + disagreement vector + verdict scalars.  
Performance ceiling for the ablation. Any experiment that matches A closely suggests its removed component was redundant; a large gap confirms the component's value.

---

**Experiment B — No Claim**  
The model must classify without knowing *what* the claim says — relying entirely on the LLM trace representations and vote statistics.  
*Expected:* Drop proportional to how much claim-specific information is non-redundant with the trace embeddings. Claims about niche entities or obscure events should be most affected. If the drop is small, traces already carry sufficient claim signal.

---

**Experiment C — No Attention Pooling**  
The attention-pooled trace representation is entirely removed. The model sees the disagreement vector (which direction traces agree in is missing) and the vote scalars, but not a synthesised "what traces collectively concluded" embedding.  
*Expected:* Clear True/False claims should suffer most, as their correct verdict is most directly encoded in the dominant trace's embedding.

---

**Experiment D — No Disagreement Vector**  
The `masked_std` component is removed.  
*Expected:* **Disproportionate drop on the Conflicting class**, because genuine disagreement between traces — captured only by the std — is the defining signal for "Conflicting" claims. Monitor class-specific F1 scores to confirm. If the drop is concentrated on Conflicting, the disagreement vector's design is validated.

---

**Experiment E — No Verdict Features**  
The 7 scalar vote-distribution features are removed.  
*Expected:* Moderate degradation. Entropy and majority-fraction encode the vote structure compactly in a form complementary to the dense disagreement vector. Some redundancy means this may be the smallest drop among all ablations.

---

### Interpreting the Attention Comparison

**Mean Pooling** treats all traces equally regardless of their quality or relevance to the claim.  
**Max Pooling** focuses on extreme activations per dimension — equivalent to selecting the "loudest" trace per feature.  
**Attention Pooling (baseline)** learns which traces to up-weight given the trace content, providing an adaptive representation.

A significant advantage of attention over mean/max confirms that trace importance is non-uniform and that the model benefits from learning to weight them selectively. Near-parity would suggest the pooled representation is dominated by trace content rather than relative weights.

---

### Notes on the Conflicting Class
The three-class setting is structurally asymmetric: True and False are mutually exclusive, while Conflicting is defined by *inter-trace disagreement*. The disagreement vector (`masked_std`) and the entropy/distinct-ratio verdict features directly target this. Experiment D is the single most diagnostic ablation for assessing whether the architecture's novelty is justified.


## 14. Conclusion

### Summary

This notebook faithfully reproduces the original Attention-MLP classifier for **CLEF CheckThat! 2026 Task 2** and extends it with a lightweight ablation study, attention comparison, and Kaggle-optimised training.

---

### What Was Preserved (Source of Truth)

| Component | Status |
|:---|:---|
| `split_and_pad` — claim row 0 / trace rows 1+, zero-pad to TRACE_MAX=20, binary mask | ✓ Exact |
| `verdict_features` — 7 scalar vote-distribution features | ✓ Exact |
| `create_train_id_to_item` — semantic-match label lookup via new_claim | ✓ Exact |
| `create_val_id_to_item` — enumerate-index mapping | ✓ Exact |
| `create_test_id_to_item` — `query_id` mapping (capital `Label` field) | ✓ Exact |
| `BaseDataset / TrainDataset / ValDataset / TestDataset` | ✓ Exact |
| `AttentionPool` — additive attention, masking, softmax, weighted sum | ✓ Exact |
| `masked_std` — element-wise std over valid traces | ✓ Exact |
| `TraceMLP` — BN → [Linear→BN→ReLU→Dropout(0.3)]×3 → Linear(3) | ✓ Exact |
| `generate_report` — confusion matrix, per-class & weighted metrics | ✓ Exact |
| Class-weighted CrossEntropyLoss | ✓ Exact |
| Adam + ReduceLROnPlateau(factor=0.5, patience=3) | ✓ Exact |
| Gradient clipping at 1.0 | ✓ Exact |
| Early stopping patience=20 | ✓ Exact |
| DataParallel when multiple GPUs | ✓ Exact |

---

### What Was Enhanced

| Enhancement | Details |
|:---|:---|
| **Embedding model** | Replaced pre-computed `.npz` with **Qwen/Qwen3-Embedding-8B** — last-token pool (verified from model card), L2-normalised, dim=4096 |
| **EMB_DIM** | Auto-detected from `model.config.hidden_size` at runtime |
| **COMBINED_DIM** | Auto-updated to `EMB_DIM×3 + SCALAR_DIM = 12295` |
| **Ablation study** | 5 experiments (A–E) systematically toggling Claim / Pool / Disagreement / Verdict |
| **Attention comparison** | 3 pooling modes: Attention (baseline), Mean, Max |
| **Mixed precision** | `torch.cuda.amp.GradScaler` + `autocast` for 2×T4 efficiency |
| **non_blocking transfers** | `tensor.to(device, non_blocking=True)` on all DataLoader tensors |
| **Reproducibility** | Seed 42 reset before every experiment; deterministic CuDNN |
| **Experiment tracking** | Accuracy, Precision, Recall, Macro F1, Weighted F1, CM per split per experiment |
| **Output files** | CSV table, full JSON, bar chart, loss curves, per-experiment confusion matrices |
| **Embedding caching** | Skip re-generation if `.npz` cache exists in `/kaggle/working/` |
| **Memory management** | Embedding model deleted + `torch.cuda.empty_cache()` before training; `gc.collect()` between experiments |

---

### Output Files in `/kaggle/working/`

| File | Description |
|:---|:---|
| `train_qwen3_embeddings.npz` | Qwen3-Embedding-8B train embeddings (cached) |
| `val_qwen3_embeddings.npz` | Qwen3-Embedding-8B val embeddings (cached) |
| `test_qwen3_embeddings.npz` | Qwen3-Embedding-8B test embeddings (cached) |
| `experiment_meta.json` | Full run configuration metadata |
| `ablation_results.csv` | Consolidated results table (sorted by Val Macro F1) |
| `ablation_results_full.json` | Per-experiment metric detail (train/val/test) |
| `ablation_macro_f1_chart.png` | Side-by-side Val + Test Macro F1 bar chart |
| `loss_curves.png` | Train/Val loss curves for all experiments |
| `{exp}_{split}_confusion_matrix.png` | Per-experiment, per-split confusion matrices (21 files) |


# REUSEEEEEEEEEEEEEEEEEEEEEEEEE

# Cell A — Setup & Expansion Module Definitions


In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPANDED VERDICT FEATURES ABLATION  (SCALAR_DIM: 7 → 1024)
# 4 methods: Learnable Projection · AE Expansion · Random Fourier · Cyclic Dup
# Reuses: train/val/test loaders, datasets, class_weights, generate_report, etc.
# ═══════════════════════════════════════════════════════════════════════════════

EXPANDED_SCALAR_DIM   = 1024
EXPANDED_COMBINED_DIM = EMB_DIM * 3 + EXPANDED_SCALAR_DIM

print("=" * 70)
print("EXPANDED VERDICT FEATURES ABLATION")
print("=" * 70)
print(f"  Original SCALAR_DIM    : {SCALAR_DIM}  →  {EXPANDED_SCALAR_DIM}")
print(f"  Original COMBINED_DIM  : {EMB_DIM * 3 + SCALAR_DIM}")
print(f"  Expanded COMBINED_DIM  : {EXPANDED_COMBINED_DIM}")
print(f"  Extra dims added       : +{EXPANDED_COMBINED_DIM - (EMB_DIM * 3 + SCALAR_DIM)}")


# ─────────────────────────────────────────────────────────────────────────────
# COLLECT VERDICT FEATURES FROM ALL SPLITS  (needed for AE pre-training)
# ─────────────────────────────────────────────────────────────────────────────
def collect_verdict_features_all_splits() -> np.ndarray:
    """Collect every 7-dim verdict vector from train + val + test."""
    vf_list = []
    for ds in [train_dataset, val_dataset, test_dataset]:
        for id_ in ds.ids:
            item = ds.id_to_item[id_]
            vf   = verdict_features(item.get(VERDICT_LIST_FIELD, []))
            vf_list.append(vf)
    arr = np.array(vf_list, dtype=np.float32)
    print(f"Collected {arr.shape[0]} verdict vectors  shape={arr.shape}  "
          f"mean={arr.mean():.4f}  std={arr.std():.4f}")
    return arr

_all_vf_np = collect_verdict_features_all_splits()


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 1 — Learnable Projection  (7 → 256 → 1024)
# Trained end-to-end with the classifier.
# ─────────────────────────────────────────────────────────────────────────────
class LearnableProjection(nn.Module):
    """Two-layer MLP: 7 → 256 → 1024.  No pre-training; trained with classifier."""
    def __init__(self,
                 in_dim:  int = SCALAR_DIM,
                 out_dim: int = EXPANDED_SCALAR_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, out_dim),
            nn.ReLU(),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)   # (B, 1024)


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 2 — Autoencoder Expansion  (7 → 256 → 1024, pre-trained then fine-tuned)
# AE is pre-trained on reconstruction of verdict features.
# The encoder (7→256→1024) is then used as the expander and fine-tuned
# jointly with the classifier (best of AE initialisation + task adaptation).
# ─────────────────────────────────────────────────────────────────────────────
class VerdictAutoencoder(nn.Module):
    """Symmetric AE: Encoder 7→256→1024, Decoder 1024→256→7."""
    def __init__(self,
                 in_dim:  int = SCALAR_DIM,
                 hidden:  int = 256,
                 latent:  int = EXPANDED_SCALAR_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, latent), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, hidden), nn.ReLU(),
            nn.Linear(hidden, in_dim),
        )
    def forward(self, x: torch.Tensor):
        z     = self.encoder(x)
        recon = self.decoder(z)
        return recon, z   # (reconstruction, latent 1024-dim)
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)


class AEEncoderExpander(nn.Module):
    """Plug-in expander: wraps a pre-trained AE encoder (7→256→1024).
    Weights are NOT frozen — they fine-tune with the main classifier."""
    def __init__(self, ae: VerdictAutoencoder):
        super().__init__()
        self.encoder = ae.encoder   # inherits pre-trained weights
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)   # (B, 1024)


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 3 — Random Fourier Features  (7 → 1024, no training)
# Approximates an RBF kernel feature map via random cosine projections.
# z(x) = sqrt(2/D) · cos(x @ W + b)   W~N(0,γ²),  b~U(0,2π)
# ─────────────────────────────────────────────────────────────────────────────
class RandomFourierFeatures(nn.Module):
    """Fixed random projections — no learnable parameters, no training needed."""
    def __init__(self,
                 in_dim:  int   = SCALAR_DIM,
                 out_dim: int   = EXPANDED_SCALAR_DIM,
                 gamma:   float = 1.0,
                 seed:    int   = 42):
        super().__init__()
        rng = np.random.RandomState(seed)
        W   = torch.tensor(
                  (rng.randn(in_dim, out_dim) * gamma).astype(np.float32))
        b   = torch.tensor(
                  rng.uniform(0, 2 * np.pi, out_dim).astype(np.float32))
        self.register_buffer("W", W)
        self.register_buffer("b", b)
        self.scale = float(np.sqrt(2.0 / out_dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.scale * torch.cos(x @ self.W + self.b)   # (B, 1024)


# ─────────────────────────────────────────────────────────────────────────────
# METHOD 4 — Cyclic Duplication  (7 → 1024, no training)
# Tiles [a,b,c,d,e,f,g, a,b,c,...] and truncates to 1024.
# repeats = ceil(1024/7) = 147  →  7×147=1029  →  [:1024]
# ─────────────────────────────────────────────────────────────────────────────
class CyclicDuplication(nn.Module):
    """Repeat verdict vector ceil(1024/7) times then slice to 1024."""
    def __init__(self,
                 in_dim:  int = SCALAR_DIM,
                 out_dim: int = EXPANDED_SCALAR_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.repeats = (out_dim + in_dim - 1) // in_dim   # = 147
        print(f"  CyclicDuplication: {in_dim} × {self.repeats} = "
              f"{in_dim*self.repeats}  →  slice to {out_dim}")
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.repeat(1, self.repeats)[:, :self.out_dim]   # (B, 1024)


# ── Quick shape verification for all 4 methods ────────────────────────────────
print("\nExpansion module shape checks:")
_dummy = torch.randn(4, SCALAR_DIM)
for _name, _mod in [
    ("LearnableProjection",  LearnableProjection()),
    ("RandomFourierFeatures",RandomFourierFeatures()),
    ("CyclicDuplication",    CyclicDuplication()),
]:
    _out = _mod(_dummy)
    assert _out.shape == (4, EXPANDED_SCALAR_DIM)
    print(f"  ✓ {_name:<25} (4,{SCALAR_DIM}) → {tuple(_out.shape)}")
del _dummy, _out, _mod
print("  [VerdictAutoencoder checked after pre-training]")

EXPANDED VERDICT FEATURES ABLATION
  Original SCALAR_DIM    : 7  →  1024
  Original COMBINED_DIM  : 12295
  Expanded COMBINED_DIM  : 13312
  Extra dims added       : +1017
Collected 10558 verdict vectors  shape=(10558, 7)  mean=0.5165  std=0.4090

Expansion module shape checks:
  CyclicDuplication: 7 × 147 = 1029  →  slice to 1024
  ✓ LearnableProjection       (4,7) → (4, 1024)
  ✓ RandomFourierFeatures     (4,7) → (4, 1024)
  ✓ CyclicDuplication         (4,7) → (4, 1024)
  [VerdictAutoencoder checked after pre-training]


# Cell B — AE Pre-training


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# AE PRE-TRAINING on verdict features  (MSE reconstruction)
# ─────────────────────────────────────────────────────────────────────────────
def pretrain_ae(verdict_np:  np.ndarray,
                n_epochs:    int   = 100,
                lr:          float = 1e-3,
                batch_size:  int   = 64,
                device:      str   = DEVICE) -> VerdictAutoencoder:
    """
    Pre-trains VerdictAutoencoder to reconstruct 7-dim verdict features.
    Returns the trained AE; its encoder is used (fine-tuned) as expander.
    """
    print("Pre-training VerdictAutoencoder ...")
    set_seed()

    ae   = VerdictAutoencoder().to(device)
    opt  = optim.Adam(ae.parameters(), lr=lr, weight_decay=1e-5)
    mse  = nn.MSELoss()
    X    = torch.tensor(verdict_np, dtype=torch.float32)
    ldr  = DataLoader(
               torch.utils.data.TensorDataset(X),
               batch_size=batch_size, shuffle=True)

    best_loss, best_state = float("inf"), None
    ae.train()
    for epoch in range(n_epochs):
        total = 0.0
        for (xb,) in ldr:
            xb = xb.to(device)
            opt.zero_grad()
            recon, _ = ae(xb)
            loss = mse(recon, xb)
            loss.backward()
            opt.step()
            total += loss.item()
        avg = total / len(ldr)
        if avg < best_loss:
            best_loss  = avg
            best_state = copy.deepcopy(ae.state_dict())
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs}  MSE: {avg:.7f}")

    ae.load_state_dict(best_state)
    ae.eval()

    # Sanity-check reconstruction quality
    with torch.no_grad():
        sample   = torch.tensor(verdict_np[:64], dtype=torch.float32).to(device)
        recon, z = ae(sample)
        print(f"\n  ✓ Reconstruction MSE  (64 samples): "
              f"{mse(recon, sample).item():.7f}")
        print(f"  ✓ Encoded latent shape            : {tuple(z.shape)}")

    print("  Pre-training complete.\n")
    return ae


_pretrained_ae = pretrain_ae(_all_vf_np, n_epochs=100, device=DEVICE)

# Verify encoder output shape
with torch.no_grad():
    _test_enc = _pretrained_ae.encode(
        torch.tensor(_all_vf_np[:4]).to(DEVICE))
    assert _test_enc.shape == (4, EXPANDED_SCALAR_DIM)
    print(f"✓ AEEncoderExpander (4,{SCALAR_DIM}) → {tuple(_test_enc.shape)}")
del _test_enc

Pre-training VerdictAutoencoder ...
  Epoch  20/100  MSE: 0.0000423
  Epoch  40/100  MSE: 0.0000288
  Epoch  60/100  MSE: 0.0000221
  Epoch  80/100  MSE: 0.0000143
  Epoch 100/100  MSE: 0.0000155

  ✓ Reconstruction MSE  (64 samples): 0.0000042
  ✓ Encoded latent shape            : (64, 1024)
  Pre-training complete.

✓ AEEncoderExpander (4,7) → (4, 1024)


# Cell C — ExpandedVerdictTraceMLP Model & Runner


In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# EXPANDED VERDICT MODEL
# Identical to TraceMLP baseline — only SCALAR_DIM changes (7 → 1024).
# ─────────────────────────────────────────────────────────────────────────────
class ExpandedVerdictTraceMLP(nn.Module):
    """
    Full model (A_Full) with verdict_expander replacing the raw 7-dim scalars.

    Architecture (everything else identical to TraceMLP baseline):
        claim (B,D) + attn_pooled (B,D) + disagreement (B,D) + expanded_verdict (B,1024)
        → cat (B, EMB_DIM*3 + 1024) → BatchNorm → [Linear→BN→ReLU→Dropout]×3 → Linear(3)
    """
    def __init__(self,
                 verdict_expander:    nn.Module,
                 expanded_scalar_dim: int  = EXPANDED_SCALAR_DIM,
                 hidden_dims:         list = None,
                 output_dim:          int  = OUTPUT_DIM):
        super().__init__()
        if hidden_dims is None: hidden_dims = HIDDEN_DIMS

        self.attn_pool        = AttentionPool(EMB_DIM)
        self.verdict_expander = verdict_expander

        combined_dim      = EMB_DIM * 3 + expanded_scalar_dim
        self.combined_dim = combined_dim

        layers = [nn.BatchNorm1d(combined_dim)]
        prev   = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        pooled           = self.attn_pool(traces, mask)
        disagreement     = masked_std(traces, mask)
        expanded_verdict = self.verdict_expander(scalars)
        x = torch.cat([claim, pooled, disagreement, expanded_verdict], dim=1)
        return self.mlp(x)


# ── Shape check ───────────────────────────────────────────────────────────────
# FIX 1: move ALL dummy tensors to DEVICE.
# AEEncoderExpander holds _pretrained_ae.encoder which is already on CUDA;
# feeding it CPU tensors is what caused the device mismatch crash.
_B3, _T3, _D3 = 2, TRACE_MAX, EMB_DIM
_c3  = torch.randn(_B3, _D3).to(DEVICE)
_tr3 = torch.randn(_B3, _T3, _D3).to(DEVICE)
_mk3 = torch.ones(_B3, _T3).to(DEVICE)
_sc3 = torch.randn(_B3, SCALAR_DIM).to(DEVICE)

for _name, _exp in [
    ("LearnableProjection",   LearnableProjection()),
    ("AEEncoderExpander",     AEEncoderExpander(_pretrained_ae)),
    ("RandomFourierFeatures", RandomFourierFeatures()),
    ("CyclicDuplication",     CyclicDuplication()),
]:
    # FIX 2: move model to DEVICE so BN/Linear weights are on CUDA too.
    _m = ExpandedVerdictTraceMLP(_exp).to(DEVICE)
    with torch.no_grad():
        _o = _m(_c3, _tr3, _mk3, _sc3)
    assert _o.shape == (_B3, OUTPUT_DIM)
    print(f"✓ ExpandedVerdictTraceMLP [{_name:<25}] "
          f"combined_dim={_m.combined_dim}  out={tuple(_o.shape)}")

del _c3, _tr3, _mk3, _sc3, _m, _o


# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT RUNNER
# ─────────────────────────────────────────────────────────────────────────────
def run_expanded_verdict_experiment(
    exp_name:         str,
    verdict_expander: nn.Module,
    train_loader, val_loader, test_loader,
    weights:          torch.Tensor,
    device:           str  = DEVICE,
    use_amp:          bool = True,
) -> dict:
    print(f"\n{'#'*72}")
    print(f"# EXPERIMENT : {exp_name}")
    print(f"# Verdict features: {SCALAR_DIM} → {EXPANDED_SCALAR_DIM} dims")
    print(f"{'#'*72}")

    set_seed()

    # FIX 3: explicitly move the expander to device BEFORE wrapping it.
    # This guarantees all sub-modules (e.g. AE encoder, RFF buffers, Linear
    # weights) land on the same device as the outer model, regardless of
    # what device they were on when created or pre-trained.
    verdict_expander = verdict_expander.to(device)
    model    = ExpandedVerdictTraceMLP(verdict_expander).to(device)

    _cdim    = model.combined_dim
    n_params = sum(p.numel() for p in model.parameters())
    print(f"combined_dim={_cdim}  params={n_params:,}")

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        print(f"DataParallel across {torch.cuda.device_count()} GPUs")

    optimizer = optim.Adam(model.parameters(),
                           lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3)
    scaler = GradScaler() if (use_amp and device.startswith("cuda")) else None

    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    train_losses, val_losses = [], []
    t0 = time.time()

    for epoch in range(MAX_ITERS):
        tr_loss              = train_epoch(model, train_loader, optimizer,
                                           criterion, device, scaler)
        va_loss, va_acc,_,_ = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        scheduler.step(va_loss)
        print(f"  Ep {epoch+1:3d}/{MAX_ITERS} | "
              f"Train {tr_loss:.4f} | Val {va_loss:.4f} | ValAcc {va_acc:.4f}")

        if va_loss < best_val_loss:
            best_val_loss, patience_counter = va_loss, 0
            _base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = copy.deepcopy(_base.state_dict())
            print("    ✓ Best model saved")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - t0
    print(f"Training time: {elapsed/60:.1f} min")

    _base = model.module if isinstance(model, nn.DataParallel) else model
    _base.load_state_dict(best_state)

    pfx = exp_name.lower().replace(" ", "_")
    tr_f,_,tr_preds,tr_labels = evaluate(model, train_loader, criterion, device)
    va_f,_,va_preds,va_labels = evaluate(model, val_loader,   criterion, device)
    te_f,_,te_preds,te_labels = evaluate(model, test_loader,  criterion, device)

    tr_m = generate_report(tr_labels, tr_preds, f"{exp_name} TRAIN", f"{pfx}_train")
    va_m = generate_report(va_labels, va_preds, f"{exp_name} VAL",   f"{pfx}_val")
    te_m = generate_report(te_labels, te_preds, f"{exp_name} TEST",  f"{pfx}_test")

    del model; gc.collect(); torch.cuda.empty_cache()

    return {
        "experiment":  exp_name,
        "config": {
            "expansion_method":    exp_name,
            "original_scalar_dim": SCALAR_DIM,
            "expanded_scalar_dim": EXPANDED_SCALAR_DIM,
            "combined_dim":        _cdim,
            "use_claim": True, "use_pool": True,
            "use_disagreement": True, "use_verdict": True,
            "pooling_mode": "attention",
        },
        "train_loss":      float(tr_f),
        "val_loss":        float(va_f),
        "test_loss":       float(te_f),
        "train_metrics":   tr_m,
        "val_metrics":     va_m,
        "test_metrics":    te_m,
        "train_losses":    [float(x) for x in train_losses],
        "val_losses":      [float(x) for x in val_losses],
        "training_time_s": float(elapsed),
    }

print("ExpandedVerdictTraceMLP and run_expanded_verdict_experiment() ready.")

  CyclicDuplication: 7 × 147 = 1029  →  slice to 1024
✓ ExpandedVerdictTraceMLP [LearnableProjection      ] combined_dim=13312  out=(2, 3)
✓ ExpandedVerdictTraceMLP [AEEncoderExpander        ] combined_dim=13312  out=(2, 3)
✓ ExpandedVerdictTraceMLP [RandomFourierFeatures    ] combined_dim=13312  out=(2, 3)
✓ ExpandedVerdictTraceMLP [CyclicDuplication        ] combined_dim=13312  out=(2, 3)
ExpandedVerdictTraceMLP and run_expanded_verdict_experiment() ready.


# Cell D — Run All 4 Experiments


In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN EXPERIMENTS F · G · H · I
# ─────────────────────────────────────────────────────────────────────────────
expanded_results = []

# F — Learnable Projection (7 → 256 → 1024, trained end-to-end)
expanded_results.append(run_expanded_verdict_experiment(
    exp_name         = "F_Proj_1024",
    verdict_expander = LearnableProjection(),
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    weights=class_weights, device=DEVICE, use_amp=True,
))

# G — Autoencoder Expansion (encoder pre-trained, fine-tuned with classifier)
expanded_results.append(run_expanded_verdict_experiment(
    exp_name         = "G_AE_1024",
    verdict_expander = AEEncoderExpander(_pretrained_ae),
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    weights=class_weights, device=DEVICE, use_amp=True,
))

# H — Random Fourier Features (fixed, no training)
expanded_results.append(run_expanded_verdict_experiment(
    exp_name         = "H_RFF_1024",
    verdict_expander = RandomFourierFeatures(gamma=1.0, seed=42),
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    weights=class_weights, device=DEVICE, use_amp=True,
))

# I — Cyclic Duplication (tile+slice, no training)
expanded_results.append(run_expanded_verdict_experiment(
    exp_name         = "I_Dup_1024",
    verdict_expander = CyclicDuplication(),
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    weights=class_weights, device=DEVICE, use_amp=True,
))

print(f"\n✓ All expanded verdict experiments complete — {len(expanded_results)} run.")


########################################################################
# EXPERIMENT : F_Proj_1024
# Verdict features: 7 → 1024 dims
########################################################################
combined_dim=13312  params=8,324,100
DataParallel across 2 GPUs
  Ep   1/100 | Train 0.9878 | Val 0.8052 | ValAcc 0.6238
    ✓ Best model saved
  Ep   2/100 | Train 0.9218 | Val 0.8308 | ValAcc 0.6244
  Ep   3/100 | Train 0.8762 | Val 0.7715 | ValAcc 0.6544
    ✓ Best model saved
  Ep   4/100 | Train 0.8278 | Val 0.7988 | ValAcc 0.6181
  Ep   5/100 | Train 0.7985 | Val 0.7984 | ValAcc 0.6294
  Ep   6/100 | Train 0.7441 | Val 0.8216 | ValAcc 0.6300
  Ep   7/100 | Train 0.6940 | Val 0.8386 | ValAcc 0.6400
  Ep   8/100 | Train 0.5873 | Val 0.8424 | ValAcc 0.6525
  Ep   9/100 | Train 0.5301 | Val 0.9344 | ValAcc 0.6531
  Ep  10/100 | Train 0.5027 | Val 0.9684 | ValAcc 0.6331
  Ep  11/100 | Train 0.4551 | Val 1.0349 | ValAcc 0.6356
  Ep  12/100 | Train 0.3891 | Val 1.0993 | ValAcc 0.642

# Cell E — Combined Results Table + Charts


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# COMBINED RESULTS TABLE  (original ablation + new expanded verdict experiments)
# ─────────────────────────────────────────────────────────────────────────────
all_results_combined = all_results + expanded_results
combined_table       = build_summary_rows(all_results_combined)

print("\n" + "=" * 112)
print("FULL RESULTS TABLE  (Ablation + Attention Comparison + Expanded Verdict)  "
      "— sorted by Val Macro F1")
print("=" * 112)
print(f"{'Experiment':<26} {'Pooling':<10} "
      f"{'ValMacF1':>9} {'ValWtF1':>9} {'ValAcc':>8} "
      f"{'TstMacF1':>9} {'TstWtF1':>9} {'TstAcc':>8}")
print("-" * 112)

_EXPANDED_NAMES = {r["experiment"] for r in expanded_results}
for row in combined_table:
    _tag = " ◀ 1024" if row["Experiment"] in _EXPANDED_NAMES else ""
    print(f"{row['Experiment']:<26} {row['Pooling']:<10}"
          f" {row['Val_Macro_F1']:>9.4f} {row['Val_Weighted_F1']:>9.4f}"
          f" {row['Val_Accuracy']:>8.4f}"
          f" {row['Test_Macro_F1']:>9.4f} {row['Test_Weighted_F1']:>9.4f}"
          f" {row['Test_Accuracy']:>8.4f}{_tag}")
print("=" * 112)

# ── Save ─────────────────────────────────────────────────────────────────────
_csv_exp  = f"{OUTPUT_DIR}/expanded_verdict_results.csv"
_json_exp = f"{OUTPUT_DIR}/expanded_verdict_results_full.json"
with open(_csv_exp, "w", newline="") as _f:
    writer = csv.DictWriter(_f, fieldnames=list(combined_table[0].keys()))
    writer.writeheader()
    writer.writerows(combined_table)
with open(_json_exp, "w") as _f:
    json.dump(numpy_to_python(all_results_combined), _f, indent=2)
print(f"\nCSV  saved → {_csv_exp}")
print(f"JSON saved → {_json_exp}")


# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — Expanded methods vs A_Full baseline  (focused comparison)
# ─────────────────────────────────────────────────────────────────────────────
_baseline = next(r for r in combined_table if r["Experiment"] == "A_Full")
_exp_rows = [r for r in combined_table if r["Experiment"] in _EXPANDED_NAMES]
_cmp_rows = [_baseline] + sorted(_exp_rows,
                                   key=lambda x: x["Val_Macro_F1"], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, title in [
    (axes[0], "Val_Macro_F1",  "Validation Macro F1"),
    (axes[1], "Test_Macro_F1", "Test Macro F1"),
]:
    names   = [r["Experiment"] for r in _cmp_rows]
    scores  = [r[key]          for r in _cmp_rows]
    colours = ["#1565C0"] + ["#2E7D32", "#1B5E20", "#66BB6A", "#A5D6A7"][:len(_cmp_rows)-1]

    bars = ax.bar(names, scores, color=colours, edgecolor="white", linewidth=0.8, width=0.55)
    ax.set_ylim(0, min(1.0, max(scores) * 1.18))
    ax.set_ylabel("Macro F1", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
    ax.axhline(_baseline[key], color="red", linestyle="--", linewidth=1.5,
               alpha=0.85, label=f"A_Full baseline ({_baseline[key]:.4f})")
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor="#1565C0", label=f"A_Full — original 7-dim"),
    Patch(facecolor="#2E7D32", label=f"Expanded 1024-dim variants"),
], loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))
plt.suptitle(
    f"Verdict Feature Expansion: {SCALAR_DIM} → {EXPANDED_SCALAR_DIM} dims\n"
    "Learnable Projection · AE Expansion · Random Fourier Features · Cyclic Duplication",
    fontsize=12, fontweight="bold", y=1.03,
)
plt.tight_layout()
_cmp_path = f"{OUTPUT_DIR}/expanded_verdict_comparison.png"
plt.savefig(_cmp_path, dpi=120, bbox_inches="tight"); plt.close()
print(f"\nFocused comparison chart saved → {_cmp_path}")


# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — Delta vs A_Full baseline  (gain / loss per method)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in [
    (axes[0], "Val_Macro_F1",  "Δ Validation Macro F1"),
    (axes[1], "Test_Macro_F1", "Δ Test Macro F1"),
]:
    deltas  = [r[key] - _baseline[key] for r in _exp_rows]
    names   = [r["Experiment"]         for r in _exp_rows]
    colours = ["#2E7D32" if d >= 0 else "#C62828" for d in deltas]

    bars = ax.bar(names, deltas, color=colours, edgecolor="white", linewidth=0.8, width=0.5)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_ylabel(f"Δ vs A_Full ({_baseline[key]:.4f})", fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.bar_label(bars, fmt="%+.4f", padding=3, fontsize=10)
    ax.tick_params(axis="x", rotation=20)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor="#2E7D32", label="Improvement over baseline"),
    Patch(facecolor="#C62828", label="Degradation vs baseline"),
], loc="lower center", ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.08))
plt.suptitle(
    f"Macro F1 Delta — Expanded Verdict ({EXPANDED_SCALAR_DIM}-dim) vs A_Full (7-dim)",
    fontsize=12, fontweight="bold", y=1.03,
)
plt.tight_layout()
_delta_path = f"{OUTPUT_DIR}/expanded_verdict_delta.png"
plt.savefig(_delta_path, dpi=120, bbox_inches="tight"); plt.close()
print(f"Delta chart saved → {_delta_path}")


# ─────────────────────────────────────────────────────────────────────────────
# PRINT CLEAN SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"EXPANDED VERDICT FEATURES — SUMMARY")
print(f"{'='*70}")
print(f"{'Method':<26} {'Val ΔMacF1':>12} {'Test ΔMacF1':>13} {'Val Acc':>10}")
print("-"*70)
for r in _exp_rows:
    dv = r["Val_Macro_F1"]  - _baseline["Val_Macro_F1"]
    dt = r["Test_Macro_F1"] - _baseline["Test_Macro_F1"]
    print(f"  {r['Experiment']:<24} {dv:>+12.4f}  {dt:>+12.4f}  {r['Val_Accuracy']:>10.4f}")
print("-"*70)
print(f"  {'A_Full (baseline 7-dim)':<24} {'---':>12}  {'---':>12}  "
      f"{_baseline['Val_Accuracy']:>10.4f}")
print("="*70)


FULL RESULTS TABLE  (Ablation + Attention Comparison + Expanded Verdict)  — sorted by Val Macro F1
Experiment                 Pooling     ValMacF1   ValWtF1   ValAcc  TstMacF1   TstWtF1   TstAcc
----------------------------------------------------------------------------------------------------------------
F_Proj_1024                attention     0.5874    0.6552   0.6544    0.5733    0.7648   0.7744 ◀ 1024
Attn_MeanPool              mean          0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MaxPool               max           0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MeanPool              mean          0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
Attn_MaxPool               max           0.5649    0.6424   0.6469    0.4542    0.6305   0.5833
E_NoVerdict                attention     0.5587    0.6296   0.6162    0.4372    0.6209   0.6083
G_AE_1024                  attention     0.5571    0.6370   0.6531    0.5613    0.7518   0.7455 ◀ 1024
A_Ful

# better look of results


In [38]:
# =============================================================================
# FINAL TEST RESULTS (All Experiments)
# Sorted by Test Average = (Macro F1 + Macro Recall)/2
# =============================================================================

all_results_combined = all_results + expanded_results

summary_rows = []

expanded_names = {r["experiment"] for r in expanded_results}

for r in all_results_combined:

    test_f1 = float(r["test_metrics"]["macro_f1"])
    test_recall = float(r["test_metrics"]["avg_recall"])

    summary_rows.append({
        "Experiment": r["experiment"],
        "Pooling": r["config"].get("pooling_mode", "-"),
        "Test_Macro_F1": test_f1,
        "Test_Macro_Recall": test_recall,
        "Test_Average": (test_f1 + test_recall) / 2,
        "Expanded": r["experiment"] in expanded_names,
    })

# Sort by average score (highest first)
summary_rows = sorted(
    summary_rows,
    key=lambda x: x["Test_Average"],
    reverse=True
)

# =============================================================================
# Print Table
# =============================================================================

print("\n" + "=" * 90)
print("FINAL TEST RESULTS (Sorted by Test Average)")
print("=" * 90)

print(
    f"{'Rank':<5}"
    f"{'Experiment':<28}"
    f"{'Macro F1':>12}"
    f"{'Macro Recall':>16}"
    f"{'Average':>12}"
)

print("-" * 90)

for idx, row in enumerate(summary_rows, start=1):

    tag = "  ◀ 1024" if row["Expanded"] else ""

    print(
        f"{idx:<5}"
        f"{row['Experiment']:<28}"
        f"{row['Test_Macro_F1']:>12.4f}"
        f"{row['Test_Macro_Recall']:>16.4f}"
        f"{row['Test_Average']:>12.4f}"
        f"{tag}"
    )

print("=" * 90)


# =============================================================================
# Save CSV
# =============================================================================

csv_path = f"{OUTPUT_DIR}/final_test_summary.csv"

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "Experiment",
            "Pooling",
            "Test_Macro_F1",
            "Test_Macro_Recall",
            "Test_Average",
            "Expanded",
        ],
    )
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"\nCSV saved -> {csv_path}")


# =============================================================================
# Save JSON
# =============================================================================

json_path = f"{OUTPUT_DIR}/final_test_summary.json"

with open(json_path, "w") as f:
    json.dump(numpy_to_python(summary_rows), f, indent=2)

print(f"JSON saved -> {json_path}")


FINAL TEST RESULTS (Sorted by Test Average)
Rank Experiment                      Macro F1    Macro Recall     Average
------------------------------------------------------------------------------------------
1    B_NoClaim                         0.5948          0.5962      0.5955
2    F_Proj_1024                       0.5733          0.5710      0.5721  ◀ 1024
3    H_RFF_1024                        0.5663          0.5638      0.5651  ◀ 1024
4    G_AE_1024                         0.5613          0.5565      0.5589  ◀ 1024
5    I_Dup_1024                        0.5571          0.5528      0.5549  ◀ 1024
6    D_NoDisagreement                  0.5152          0.5431      0.5291
7    C_NoAttention                     0.5184          0.5306      0.5245
8    A_Full                            0.5075          0.5205      0.5140
9    Attn_MeanPool                     0.4542          0.4719      0.4630
10   Attn_MaxPool                      0.4542          0.4719      0.4630
11   Attn_MeanPool

In [39]:
print(0)

0


# combine ablation toggles + 1024-dim verdict


In [40]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURABLE EXPANDED MODEL — combine ablation toggles + 1024-dim verdict
# ═══════════════════════════════════════════════════════════════

class ConfigurableExpandedTraceMLP(nn.Module):
    """
    Like ConfigurableTraceMLP, but verdict scalars go through verdict_expander
    (7 -> 1024) instead of raw 7-dim. Supports use_pool / use_disagreement toggle.
    use_claim and use_verdict always True here (claim+verdict always kept).
    """
    def __init__(self, exp_config: dict, verdict_expander: nn.Module,
                 emb_dim=None, expanded_scalar_dim=EXPANDED_SCALAR_DIM,
                 hidden_dims=None, output_dim=OUTPUT_DIM):
        super().__init__()
        if emb_dim is None: emb_dim = EMB_DIM
        if hidden_dims is None: hidden_dims = HIDDEN_DIMS

        self.use_pool  = exp_config.get("use_pool", True)
        self.use_disag = exp_config.get("use_disagreement", True)
        self.verdict_expander = verdict_expander

        if self.use_pool:
            self.attn_pool = AttentionPool(emb_dim)

        combined_dim = emb_dim  # claim always
        if self.use_pool:  combined_dim += emb_dim
        if self.use_disag: combined_dim += emb_dim
        combined_dim += expanded_scalar_dim  # verdict always
        self.combined_dim = combined_dim

        layers = [nn.BatchNorm1d(combined_dim)]
        prev = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        parts = [claim]
        if self.use_pool:
            parts.append(self.attn_pool(traces, mask))
        if self.use_disag:
            parts.append(masked_std(traces, mask))
        parts.append(self.verdict_expander(scalars))
        return self.mlp(torch.cat(parts, dim=1))


# shape check
_c4  = torch.randn(2, EMB_DIM).to(DEVICE)
_tr4 = torch.randn(2, TRACE_MAX, EMB_DIM).to(DEVICE)
_mk4 = torch.ones(2, TRACE_MAX).to(DEVICE)
_sc4 = torch.randn(2, SCALAR_DIM).to(DEVICE)
_test_cfg = dict(use_pool=False, use_disagreement=True)
_m4 = ConfigurableExpandedTraceMLP(_test_cfg, CyclicDuplication()).to(DEVICE)
print("✓ combined_dim:", _m4.combined_dim, "out:", _m4(_c4,_tr4,_mk4,_sc4).shape)
del _c4,_tr4,_mk4,_sc4,_m4

  CyclicDuplication: 7 × 147 = 1029  →  slice to 1024
✓ combined_dim: 9216 out: torch.Size([2, 3])


In [41]:
def run_configurable_expanded_experiment(exp_name, exp_config, verdict_expander,
                                         train_loader, val_loader, test_loader,
                                         weights, device=DEVICE, use_amp=True):
    print(f"\n{'#'*72}\n# EXPERIMENT : {exp_name}\n{'#'*72}")
    set_seed()
    verdict_expander = verdict_expander.to(device)
    model = ConfigurableExpandedTraceMLP(exp_config, verdict_expander).to(device)
    _cdim = model.combined_dim
    print(f"combined_dim={_cdim}  params={sum(p.numel() for p in model.parameters()):,}")

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = GradScaler() if (use_amp and device.startswith("cuda")) else None

    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    train_losses, val_losses = [], []
    t0 = time.time()

    for epoch in range(MAX_ITERS):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        va_loss, va_acc,_,_ = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr_loss); val_losses.append(va_loss)
        scheduler.step(va_loss)
        print(f"  Ep {epoch+1:3d}/{MAX_ITERS} | Train {tr_loss:.4f} | Val {va_loss:.4f} | ValAcc {va_acc:.4f}")
        if va_loss < best_val_loss:
            best_val_loss, patience_counter = va_loss, 0
            _base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = copy.deepcopy(_base.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    elapsed = time.time() - t0
    _base = model.module if isinstance(model, nn.DataParallel) else model
    _base.load_state_dict(best_state)

    pfx = exp_name.lower().replace(" ", "_")
    tr_f,_,tr_p,tr_l = evaluate(model, train_loader, criterion, device)
    va_f,_,va_p,va_l = evaluate(model, val_loader,   criterion, device)
    te_f,_,te_p,te_l = evaluate(model, test_loader,  criterion, device)

    tr_m = generate_report(tr_l, tr_p, f"{exp_name} TRAIN", f"{pfx}_train")
    va_m = generate_report(va_l, va_p, f"{exp_name} VAL",   f"{pfx}_val")
    te_m = generate_report(te_l, te_p, f"{exp_name} TEST",  f"{pfx}_test")

    del model; gc.collect(); torch.cuda.empty_cache()

    return {
        "experiment": exp_name,
        "config": {**exp_config, "expanded_scalar_dim": EXPANDED_SCALAR_DIM, "combined_dim": _cdim},
        "train_loss": float(tr_f), "val_loss": float(va_f), "test_loss": float(te_f),
        "train_metrics": tr_m, "val_metrics": va_m, "test_metrics": te_m,
        "train_losses": [float(x) for x in train_losses],
        "val_losses": [float(x) for x in val_losses],
        "training_time_s": float(elapsed),
    }

print("run_configurable_expanded_experiment() ready.")

run_configurable_expanded_experiment() ready.


In [42]:
CFG_C = dict(use_pool=False, use_disagreement=True)   # No Attention
CFG_D = dict(use_pool=True,  use_disagreement=False)  # No Disagreement

EXPANDERS = {
    "Proj": LearnableProjection,
    "AE":   lambda: AEEncoderExpander(_pretrained_ae),
    "RFF":  lambda: RandomFourierFeatures(gamma=1.0, seed=42),
    "Dup":  CyclicDuplication,
}

expanded_ablation_results = []

for _ename, _efactory in EXPANDERS.items():
    expanded_ablation_results.append(run_configurable_expanded_experiment(
        exp_name=f"C_NoAttn_{_ename}_1024", exp_config=CFG_C,
        verdict_expander=_efactory(),
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights,
    ))

for _ename, _efactory in EXPANDERS.items():
    expanded_ablation_results.append(run_configurable_expanded_experiment(
        exp_name=f"D_NoDisag_{_ename}_1024", exp_config=CFG_D,
        verdict_expander=_efactory(),
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights,
    ))

print(f"\n✓ {len(expanded_ablation_results)} new experiments done.")
all_results_combined += expanded_ablation_results


########################################################################
# EXPERIMENT : C_NoAttn_Proj_1024
########################################################################
combined_dim=9216  params=5,169,667
  Ep   1/100 | Train 0.9790 | Val 0.8194 | ValAcc 0.6162
  Ep   2/100 | Train 0.9187 | Val 0.8274 | ValAcc 0.6000
  Ep   3/100 | Train 0.8567 | Val 0.7937 | ValAcc 0.6300
  Ep   4/100 | Train 0.8182 | Val 0.8117 | ValAcc 0.6150
  Ep   5/100 | Train 0.8059 | Val 0.7999 | ValAcc 0.6369
  Ep   6/100 | Train 0.7421 | Val 0.8203 | ValAcc 0.6381
  Ep   7/100 | Train 0.7074 | Val 0.8999 | ValAcc 0.5825
  Ep   8/100 | Train 0.5831 | Val 0.9111 | ValAcc 0.6225
  Ep   9/100 | Train 0.5268 | Val 0.9173 | ValAcc 0.6375
  Ep  10/100 | Train 0.4732 | Val 0.9563 | ValAcc 0.6412
  Ep  11/100 | Train 0.4544 | Val 0.9791 | ValAcc 0.6394
  Ep  12/100 | Train 0.3750 | Val 1.0456 | ValAcc 0.6450
  Ep  13/100 | Train 0.3583 | Val 1.0981 | ValAcc 0.6375
  Ep  14/100 | Train 0.3488 | Val 1.1398 |

In [43]:
# =============================================================================
# FINAL TEST RESULTS (All Experiments — Ablation + Attention + Expanded + C/D combos)
# Sorted by Test Average = (Macro F1 + Macro Recall)/2
# =============================================================================

# Combine ALL result lists generated so far (dedup-safe via dict keyed by name)
_all_runs = {}
for _r in (all_results + expanded_results + expanded_ablation_results):
    _all_runs[_r["experiment"]] = _r   # later entries overwrite duplicates
all_results_combined = list(_all_runs.values())

_expanded_names = {r["experiment"] for r in expanded_results}
_expanded_ablation_names = {r["experiment"] for r in expanded_ablation_results}

summary_rows = []
for r in all_results_combined:
    test_f1     = float(r["test_metrics"]["macro_f1"])
    test_recall = float(r["test_metrics"]["avg_recall"])
    name = r["experiment"]
    summary_rows.append({
        "Experiment":        name,
        "Pooling":           r["config"].get("pooling_mode", "-"),
        "Test_Macro_F1":     test_f1,
        "Test_Macro_Recall": test_recall,
        "Test_Average":      (test_f1 + test_recall) / 2,
        "Expanded":          name in _expanded_names or name in _expanded_ablation_names,
    })

summary_rows = sorted(summary_rows, key=lambda x: x["Test_Average"], reverse=True)

# =============================================================================
# Print Table
# =============================================================================
print("\n" + "=" * 90)
print(f"FINAL TEST RESULTS — {len(summary_rows)} experiments (Sorted by Test Average)")
print("=" * 90)
print(f"{'Rank':<5}{'Experiment':<28}{'Macro F1':>12}{'Macro Recall':>16}{'Average':>12}")
print("-" * 90)
for idx, row in enumerate(summary_rows, start=1):
    tag = "  ◀ 1024" if row["Expanded"] else ""
    print(f"{idx:<5}{row['Experiment']:<28}"
          f"{row['Test_Macro_F1']:>12.4f}{row['Test_Macro_Recall']:>16.4f}"
          f"{row['Test_Average']:>12.4f}{tag}")
print("=" * 90)

# =============================================================================
# Save CSV
# =============================================================================
csv_path = f"{OUTPUT_DIR}/final_test_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "Experiment", "Pooling", "Test_Macro_F1",
        "Test_Macro_Recall", "Test_Average", "Expanded",
    ])
    writer.writeheader()
    writer.writerows(summary_rows)
print(f"\nCSV saved -> {csv_path}")

# =============================================================================
# Save JSON
# =============================================================================
json_path = f"{OUTPUT_DIR}/final_test_summary.json"
with open(json_path, "w") as f:
    json.dump(numpy_to_python(summary_rows), f, indent=2)
print(f"JSON saved -> {json_path}")


FINAL TEST RESULTS — 19 experiments (Sorted by Test Average)
Rank Experiment                      Macro F1    Macro Recall     Average
------------------------------------------------------------------------------------------
1    B_NoClaim                         0.5948          0.5962      0.5955
2    C_NoAttn_Dup_1024                 0.5971          0.5857      0.5914  ◀ 1024
3    D_NoDisag_Dup_1024                0.5810          0.5861      0.5835  ◀ 1024
4    C_NoAttn_AE_1024                  0.5779          0.5868      0.5823  ◀ 1024
5    D_NoDisag_AE_1024                 0.5714          0.5861      0.5787  ◀ 1024
6    D_NoDisag_RFF_1024                0.5769          0.5770      0.5769  ◀ 1024
7    F_Proj_1024                       0.5733          0.5710      0.5721  ◀ 1024
8    H_RFF_1024                        0.5663          0.5638      0.5651  ◀ 1024
9    G_AE_1024                         0.5613          0.5565      0.5589  ◀ 1024
10   I_Dup_1024                        0.55

In [44]:
print("end")

end


# XGBOOST.................

In [45]:
import xgboost as xgb
from sklearn.metrics import f1_score, recall_score

# Fixed (untrained, seeded) attention pool — shared across all 3 expander runs
# so pooled-trace features stay identical; only verdict_expander differs.
set_seed()
_xgb_attn_pool = AttentionPool(EMB_DIM).to(DEVICE).eval()

def extract_xgb_features(loader, verdict_expander, attn_pool=_xgb_attn_pool, device=DEVICE):
    """Builds (X, y) using claim + attn_pooled + disagreement + expanded_verdict(1024)."""
    verdict_expander = verdict_expander.to(device).eval()
    attn_pool = attn_pool.to(device).eval()
    X_list, y_list = [], []
    with torch.no_grad():
        for claim, traces, mask, scalars, labels in loader:
            claim   = claim.to(device, non_blocking=True)
            traces  = traces.to(device, non_blocking=True)
            mask    = mask.to(device, non_blocking=True)
            scalars = scalars.to(device, non_blocking=True)

            pooled      = attn_pool(traces, mask)
            disag       = masked_std(traces, mask)
            exp_verdict = verdict_expander(scalars)

            feat = torch.cat([claim, pooled, disag, exp_verdict], dim=1)
            X_list.append(feat.cpu().numpy())
            y_list.append(labels.numpy())

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    return X, y

print(f"Feature dim per sample: {EMB_DIM*3 + EXPANDED_SCALAR_DIM}")
print(f"  = claim({EMB_DIM}) + pooled({EMB_DIM}) + disagreement({EMB_DIM}) + verdict({EXPANDED_SCALAR_DIM})")

Feature dim per sample: 13312
  = claim(4096) + pooled(4096) + disagreement(4096) + verdict(1024)


In [46]:
XGB_EXPANDERS = {
    "Proj": LearnableProjection,                          # untrained → fixed random projection (no backprop here)
    "AE":   lambda: AEEncoderExpander(_pretrained_ae),     # uses your already pre-trained AE encoder
    "Dup":  CyclicDuplication,                             # deterministic tiling, no training needed
}

xgb_results = []

for _ename, _efactory in XGB_EXPANDERS.items():
    print(f"\n{'#'*72}\n# XGBOOST — Verdict Expansion: {_ename}\n{'#'*72}")
    set_seed()
    expander = _efactory()

    print(f"[{_ename}] Extracting TRAIN features...")
    X_train, y_train = extract_xgb_features(train_loader, expander)
    print(f"[{_ename}] Extracting VAL features...")
    X_val, y_val     = extract_xgb_features(val_loader, expander)
    print(f"[{_ename}] Extracting TEST features...")
    X_test, y_test   = extract_xgb_features(test_loader, expander)

    print(f"[{_ename}] X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")

    # Convert your existing class_weights tensor -> per-sample weights for XGBoost
    _cw = class_weights.cpu().numpy()
    sample_weight = _cw[y_train]

    clf = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=OUTPUT_DIM,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda" if DEVICE == "cuda" else "cpu",
        early_stopping_rounds=20,
        eval_metric="mlogloss",
        random_state=RANDOM_SEED,
    )

    clf.fit(
        X_train, y_train,
        sample_weight=sample_weight,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )

    y_pred = clf.predict(X_test)

    macro_f1     = f1_score(y_test, y_pred, average="macro", zero_division=0)
    macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    avg_score    = (macro_f1 + macro_recall) / 2

    print(f"\n[{_ename}] Test Macro F1     : {macro_f1:.4f}")
    print(f"[{_ename}] Test Macro Recall : {macro_recall:.4f}")
    print(f"[{_ename}] Average            : {avg_score:.4f}")

    xgb_results.append({
        "Experiment": f"XGB_{_ename}_1024",
        "Test_Macro_F1": float(macro_f1),
        "Test_Macro_Recall": float(macro_recall),
        "Test_Average": float(avg_score),
    })

    del clf, X_train, X_val, X_test, expander
    gc.collect()

# ── Summary table ─────────────────────────────────────────────
xgb_results = sorted(xgb_results, key=lambda x: x["Test_Average"], reverse=True)

print("\n" + "=" * 70)
print("XGBOOST RESULTS — 1024-dim Expanded Verdict Features (Test Set)")
print("=" * 70)
print(f"{'Experiment':<22}{'Macro F1':>12}{'Macro Recall':>16}{'Average':>12}")
print("-" * 70)
for row in xgb_results:
    print(f"{row['Experiment']:<22}{row['Test_Macro_F1']:>12.4f}"
          f"{row['Test_Macro_Recall']:>16.4f}{row['Test_Average']:>12.4f}")
print("=" * 70)

# ── Save ──────────────────────────────────────────────────────
with open(f"{OUTPUT_DIR}/xgb_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Experiment","Test_Macro_F1","Test_Macro_Recall","Test_Average"])
    writer.writeheader()
    writer.writerows(xgb_results)
with open(f"{OUTPUT_DIR}/xgb_results.json", "w") as f:
    json.dump(xgb_results, f, indent=2)
print(f"\nSaved -> {OUTPUT_DIR}/xgb_results.csv / .json")


########################################################################
# XGBOOST — Verdict Expansion: Proj
########################################################################
[Proj] Extracting TRAIN features...
[Proj] Extracting VAL features...
[Proj] Extracting TEST features...
[Proj] X_train=(6400, 13312)  X_val=(1600, 13312)  X_test=(2558, 13312)
[0]	validation_0-mlogloss:1.08373
[50]	validation_0-mlogloss:0.82171
[82]	validation_0-mlogloss:0.83504

[Proj] Test Macro F1     : 0.5303
[Proj] Test Macro Recall : 0.5088
[Proj] Average            : 0.5195

########################################################################
# XGBOOST — Verdict Expansion: AE
########################################################################
[AE] Extracting TRAIN features...
[AE] Extracting VAL features...
[AE] Extracting TEST features...
[AE] X_train=(6400, 13312)  X_val=(1600, 13312)  X_test=(2558, 13312)
[0]	validation_0-mlogloss:1.08145
[50]	validation_0-mlogloss:0.91353
[52]	validati

In [47]:
# =============================================================================
# FINAL XGBOOST RESULTS (Proj / AE / Dup — 1024-dim Verdict Features)
# Sorted by Test Average = (Macro F1 + Macro Recall)/2
# =============================================================================

xgb_results = sorted(xgb_results, key=lambda x: x["Test_Average"], reverse=True)

print("\n" + "=" * 90)
print(f"FINAL XGBOOST RESULTS — {len(xgb_results)} experiments (Sorted by Test Average)")
print("=" * 90)
print(f"{'Rank':<5}{'Experiment':<28}{'Macro F1':>12}{'Macro Recall':>16}{'Average':>12}")
print("-" * 90)
for idx, row in enumerate(xgb_results, start=1):
    print(f"{idx:<5}{row['Experiment']:<28}"
          f"{row['Test_Macro_F1']:>12.4f}{row['Test_Macro_Recall']:>16.4f}"
          f"{row['Test_Average']:>12.4f}")
print("=" * 90)

# ── Save CSV ──────────────────────────────────────────────────
csv_path = f"{OUTPUT_DIR}/final_xgb_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Experiment", "Test_Macro_F1", "Test_Macro_Recall", "Test_Average"])
    writer.writeheader()
    writer.writerows(xgb_results)
print(f"\nCSV saved -> {csv_path}")

# ── Save JSON ─────────────────────────────────────────────────
json_path = f"{OUTPUT_DIR}/final_xgb_summary.json"
with open(json_path, "w") as f:
    json.dump(numpy_to_python(xgb_results), f, indent=2)
print(f"JSON saved -> {json_path}")


FINAL XGBOOST RESULTS — 3 experiments (Sorted by Test Average)
Rank Experiment                      Macro F1    Macro Recall     Average
------------------------------------------------------------------------------------------
1    XGB_AE_1024                       0.5652          0.5445      0.5549
2    XGB_Dup_1024                      0.5468          0.5256      0.5362
3    XGB_Proj_1024                     0.5303          0.5088      0.5195

CSV saved -> /kaggle/working/final_xgb_summary.csv
JSON saved -> /kaggle/working/final_xgb_summary.json
